In [2]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 1: unified patient-grouped folds shared by BOTH stages
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
D="/root/autodl-tmp/CBIS"; NFOLD=5; SEED=42

for LES,CSV in [("mass","cbis_mass_fixed.csv"),("calc","cbis_calc_fixed.csv")]:
    d=pd.read_csv(os.path.join(D,CSV))
    d["label"]=d["label"].astype(int)
    d["assessment"]=pd.to_numeric(d["assessment"],errors="coerce")
    d=d.dropna(subset=["img","msk","label"]).reset_index(drop=True)

    sc=[c for c in ["side","left or right breast"] if c in d.columns][0]
    lc=[c for c in ["lesion","abnormality id"] if c in d.columns][0]
    d["lesion_key"]=d.patient_id.astype(str)+"_"+d[sc].astype(str)+"_"+d[lc].astype(str)

    strat=d.label.astype(str)+"_"+d.assessment.isin([3,4]).astype(int).astype(str)
    d["fold"]=-1
    for k,(tr,te) in enumerate(StratifiedGroupKFold(NFOLD,shuffle=True,random_state=SEED)
                               .split(d,strat,d.patient_id.values)):
        d.loc[te,"fold"]=k

    # per-fold role: train / val / test  (val = 12% of that fold's training patients)
    for k in range(NFOLD):
        col="role_f"+str(k)
        d[col]="train"
        d.loc[d.fold==k,col]="test"
        trp=sorted(set(d.loc[d.fold!=k,"patient_id"]))
        rs=np.random.RandomState(1000+k)
        vp=set(rs.permutation(np.array(trp,dtype=object))[:max(1,int(0.12*len(trp)))])
        d.loc[(d.fold!=k)&(d.patient_id.isin(vp)),col]="val"

    out=os.path.join(D,"unified_folds_"+LES+".csv")
    d.to_csv(out,index=False)

    print("="*76); print(LES.upper()+"   n="+str(len(d))+"   patients="+str(d.patient_id.nunique())+
          "   lesions="+str(d.lesion_key.nunique())); print("="*76)
    print("  fold   n_img  n_pat  n_les  malig%   B4%   | train/val/test images")
    for k in range(NFOLD):
        q=d[d.fold==k]; r=d["role_f"+str(k)]
        print("   "+str(k)+"     "+str(len(q)).rjust(5)+"  "+str(q.patient_id.nunique()).rjust(5)+
              "  "+str(q.lesion_key.nunique()).rjust(5)+"   "+format(100*q.label.mean(),".1f").rjust(5)+
              "%  "+format(100*(q.assessment==4).mean(),".0f").rjust(3)+"%   | "+
              str((r=="train").sum())+"/"+str((r=="val").sum())+"/"+str((r=="test").sum()))

    bad=[]
    for k in range(NFOLD):
        r=d["role_f"+str(k)]
        trp=set(d.loc[r=="train","patient_id"]); vap=set(d.loc[r=="val","patient_id"]); tep=set(d.loc[r=="test","patient_id"])
        if (trp&tep) or (vap&tep) or (trp&vap): bad.append(k)
    print("  leakage check: "+("PASS - no patient appears in two roles in any fold"
                               if not bad else "FAIL in folds "+str(bad)))
    print("  saved "+os.path.basename(out))

MASS   n=1696   patients=892   lesions=1005
  fold   n_img  n_pat  n_les  malig%   B4%   | train/val/test images
   0       339    178    199    46.3%   41%   | 1202/155/339
   1       340    178    201    46.2%   41%   | 1184/172/340
   2       339    179    201    46.3%   42%   | 1192/165/339
   3       339    176    199    46.0%   41%   | 1195/162/339
   4       339    181    205    46.3%   42%   | 1203/154/339
  leakage check: PASS - no patient appears in two roles in any fold
  saved unified_folds_mass.csv
CALC   n=1866   patients=753   lesions=1042
  fold   n_img  n_pat  n_les  malig%   B4%   | train/val/test images
   0       374    149    206    36.1%   49%   | 1303/189/374
   1       373    150    209    35.9%   52%   | 1335/158/373
   2       373    152    208    36.2%   50%   | 1301/192/373
   3       373    151    210    35.9%   50%   | 1308/185/373
   4       373    151    209    35.9%   49%   | 1283/210/373
  leakage check: PASS - no patient appears in two roles in any fo

In [2]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 2 — SEGMENTATION on UNIFIED FOLDS  (mass + calcification)
#
# ARCHITECTURE (unchanged from your DS-Attn-UNet+ASPP cell)
#   • Encoder      : 4 levels, double-conv blocks (32→64→128→256), MaxPool2d(2)
#   • Bottleneck   : ASPP — atrous pyramid, dilations 6/12/18 + global pooling,
#                    5 branches concatenated then projected (multi-scale context)
#   • Attention    : AG gates on every skip connection (Wg + Wx → ReLU → psi → Sigmoid)
#   • Decoder      : 4 levels, ConvTranspose2d upsampling, attention-gated skips
#   • Heads        : main out (full res) + ds2/ds3/ds4 deep-supervision heads
#                    (deep supervision active in training only)
#
# PREPROCESSING
#   • CLAHE clipLimit=2.0, tile 8×8       (no Gaussian blur in this variant)
#   • images resized to 256×256, masks nearest-neighbour
#
# AUGMENTATION — 8× lockstep (image and mask transformed identically)
#   0 identity | 1 hflip | 2 vflip | 3 rot90 | 4 rot180 | 5 rot270
#   6 rotate ±25° + scale 0.9–1.1 | 7 brightness ×0.85–1.15 (image only)
#
# LOSS
#   • tversky_ce = 0.3·CrossEntropy + 0.7·(1 − Tversky), TV_A=0.7 TV_B=0.3
#   • deep supervision weights [1.0, 0.5, 0.3, 0.2] on main/d2/d3/d4
#   • targets downsampled with nearest interpolation for the ds heads
#
# OPTIMISATION
#   • Adam lr=1e-3 | ReduceLROnPlateau(mode=max, patience=4, factor=0.5)
#   • AMP mixed precision | batch 16 | max 60 epochs | early stop patience 10
#   • model selection on best VALIDATION Dice
#
# EVALUATION
#   • threshold 0.5 | Dice, IoU, precision, recall (smoothed +1)
#
# WHAT IS NEW HERE (and only this)
#   • runs 5 patient-grouped folds from unified_folds_{LES}.csv
#   • each fold: train / val / test roles taken from role_f{k}
#   • leakage assertions before every fold
#   • predicted masks for each fold's UNSEEN test patients saved at 512×512
#     into predmasks_{LES}/  → consumed by Phase 3 classification
#   • per-fold checkpoints seg_dsaspp_{LES}_fold{k}.pth
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3; MULT=8; THR=0.5; TV_A,TV_B=0.7,0.3
DS_WEIGHTS=[1.0,0.5,0.3,0.2]
torch.backends.cudnn.benchmark=True

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,d,aug,mult=1): s.df=d.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))

class ASPP(nn.Module):
    def __init__(s,i,o):
        super().__init__()
        s.b0=nn.Sequential(nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b1=nn.Sequential(nn.Conv2d(i,o,3,padding=6,dilation=6),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b2=nn.Sequential(nn.Conv2d(i,o,3,padding=12,dilation=12),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b3=nn.Sequential(nn.Conv2d(i,o,3,padding=18,dilation=18),nn.BatchNorm2d(o),nn.ReLU(True))
        s.gp=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.proj=nn.Sequential(nn.Conv2d(o*5,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(s,x):
        g=F.interpolate(s.gp(x),size=x.shape[2:],mode="bilinear",align_corners=False)
        return s.proj(torch.cat([s.b0(x),s.b1(x),s.b2(x),s.b3(x),g],1))

class DSAttnUNet(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2)
        s.bn=ASPP(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out =nn.Conv2d(b,2,1)
        s.ds2=nn.Conv2d(b*2,2,1); s.ds3=nn.Conv2d(b*4,2,1); s.ds4=nn.Conv2d(b*8,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        main=s.out(d1)
        if s.training: return main, s.ds2(d2), s.ds3(d3), s.ds4(d4)
        return main

def tversky_ce(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    return 0.3*ce+0.7*(1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean())

def ds_loss(outs,t):
    main,o2,o3,o4=outs
    L=DS_WEIGHTS[0]*tversky_ce(main,t)
    for w,o in zip(DS_WEIGHTS[1:],[o2,o3,o4]):
        td=F.interpolate(t.unsqueeze(1).float(),size=o.shape[2:],mode="nearest").squeeze(1).long()
        L=L+w*tversky_ce(o,td)
    return L

@torch.no_grad()
def sc(net,d):
    net.eval(); ld=DataLoader(DS(d,False),batch_size=BATCH,shuffle=False,num_workers=0); r=[]
    for x,y,_ in ld:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r.append(dict(dice=(2*tp+1)/(2*tp+fp+fn+1),iou=(tp+1)/(tp+fp+fn+1),
                          prec=tp/(tp+fp+1e-9),rec=tp/(tp+fn+1e-9)))
    R=pd.DataFrame(r)
    return dict(n=len(R),dice=R.dice.mean(),median=R.dice.median(),iou=R.iou.mean(),
                prec=R.prec.mean(),rec=R.rec.mean())

def run(LES, PREV):
    OUT=os.path.join(D,"predmasks_"+LES); os.makedirs(OUT,exist_ok=True)
    d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv")).reset_index(drop=True)
    print("\n"+"#"*74); print("#  "+LES.upper()+"   n="+str(len(d))+
          "   patients="+str(d.patient_id.nunique())+"   crops: "+d.img.iloc[0].split("/")[-2])
    print("#"*74)
    oof=np.zeros(len(d)); folds=[]
    for k in range(5):
        role=d["role_f"+str(k)]
        tr=d[role=="train"]; va=d[role=="val"]; te=d[role=="test"]
        assert len(set(tr.patient_id)&set(te.patient_id))==0, "LEAK train/test fold "+str(k)
        assert len(set(va.patient_id)&set(te.patient_id))==0, "LEAK val/test fold "+str(k)
        assert len(set(tr.patient_id)&set(va.patient_id))==0, "LEAK train/val fold "+str(k)
        print("\n### fold "+str(k)+" | train "+str(len(tr))+" x"+str(MULT)+
              " | val "+str(len(va))+" | test "+str(len(te)))
        t0=time.time(); torch.manual_seed(k); np.random.seed(k)
        tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
        net=DSAttnUNet().to(DEV); scaler=torch.amp.GradScaler()
        opt=torch.optim.Adam(net.parameters(),lr=LR)
        sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)
        best,bs,ni=0.,None,0
        for ep in range(1,EPOCHS+1):
            net.train(); tot=0.; nb=0
            for x,y,_ in tl:
                x=x.to(DEV); y=y.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"): outs=net(x); l=ds_loss(outs,y)
                scaler.scale(l).backward(); scaler.step(opt); scaler.update()
                tot+=l.item(); nb+=1
            vd=sc(net,va)["dice"]; sch.step(vd)
            if vd>best: best=vd; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0
            else: ni+=1
            print("  ep "+str(ep).rjust(2)+" | loss "+format(tot/max(nb,1),".4f")+
                  " | val-Dice "+format(vd,".4f")+(" *" if vd==best else ""))
            if ni>=10: print("  early stop"); break
        net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
        torch.save({q:v.cpu() for q,v in net.state_dict().items()},
                   os.path.join(D,"seg_dsaspp_"+LES+"_fold"+str(k)+".pth"))
        m=sc(net,te); folds.append(m["dice"])
        print("  TEST Dice "+format(m["dice"],".4f")+" (median "+format(m["median"],".4f")+
              ") | IoU "+format(m["iou"],".4f")+" | P "+format(m["prec"],".3f")+
              " | R "+format(m["rec"],".3f")+" | "+format(time.time()-t0,".0f")+"s")

        # predicted masks for THIS fold's unseen test patients -> 512px for Phase 3
        net.eval(); tei=te.index.values
        with torch.no_grad():
            for x,y,jj in DataLoader(DS(te,False),batch_size=BATCH,shuffle=False,num_workers=0):
                x=x.to(DEV)
                with torch.amp.autocast(device_type="cuda"): o=net(x)
                pr=torch.softmax(o.float(),1)[:,1].cpu().numpy()
                for i in range(len(jj)):
                    gi=int(tei[int(jj[i])]); mm=(pr[i]>THR).astype(np.uint8)
                    gt=cv2.imread(d.iloc[gi]["msk"],cv2.IMREAD_GRAYSCALE)
                    gt=(cv2.resize(gt,(IMG,IMG),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
                    tp=(mm*gt).sum(); fp=(mm*(1-gt)).sum(); fn=((1-mm)*gt).sum()
                    oof[gi]=(2*tp+1)/(2*tp+fp+fn+1)
                    cv2.imwrite(os.path.join(OUT,os.path.basename(d.iloc[gi]["img"]).replace("_img.png","")+"_pred.png"),
                                cv2.resize(mm*255,(512,512),interpolation=cv2.INTER_NEAREST))
    d["oof_dice"]=oof
    d.to_csv(os.path.join(D,"unified_folds_"+LES+".csv"),index=False)
    print("\n  "+LES.upper()+" folds: "+str([round(x,4) for x in folds]))
    print("  MEAN "+format(np.mean(folds),".4f")+" ± "+format(np.std(folds),".4f")+
          "   pooled OOF "+format(oof.mean(),".4f")+"   (single-split baseline: "+PREV+")")
    print("  masks -> "+OUT)
    return folds

r_mass=run("mass","0.9238")
r_calc=run("calc","0.8843")
print("\n"+"="*74)
print("PHASE 2 COMPLETE — DS-Attn-UNet + ASPP on unified patient-grouped folds")
print("="*74)
print("  MASS          "+format(np.mean(r_mass),".4f")+" ± "+format(np.std(r_mass),".4f")+"   (single-split: 0.9238)")
print("  CALCIFICATION "+format(np.mean(r_calc),".4f")+" ± "+format(np.std(r_calc),".4f")+"   (single-split: 0.8843)")
print("  next: Phase 3 — classification guided by predmasks_mass / predmasks_calc")
print("="*74)


##########################################################################
#  MASS   n=1696   patients=892   crops: crops_fixed_mass
##########################################################################

### fold 0 | train 1202 x8 | val 155 | test 339
  ep  1 | loss 0.3560 | val-Dice 0.8714 *
  ep  2 | loss 0.2856 | val-Dice 0.8787 *
  ep  3 | loss 0.2682 | val-Dice 0.8659
  ep  4 | loss 0.2569 | val-Dice 0.8905 *
  ep  5 | loss 0.2458 | val-Dice 0.8853
  ep  6 | loss 0.2383 | val-Dice 0.8811
  ep  7 | loss 0.2306 | val-Dice 0.8846
  ep  8 | loss 0.2208 | val-Dice 0.8955 *
  ep  9 | loss 0.2161 | val-Dice 0.9006 *
  ep 10 | loss 0.2083 | val-Dice 0.8807
  ep 11 | loss 0.2025 | val-Dice 0.8908
  ep 12 | loss 0.1959 | val-Dice 0.8859
  ep 13 | loss 0.1911 | val-Dice 0.8997
  ep 14 | loss 0.1822 | val-Dice 0.8767
  ep 15 | loss 0.1654 | val-Dice 0.9010 *
  ep 16 | loss 0.1592 | val-Dice 0.9008
  ep 17 | loss 0.1545 | val-Dice 0.9014 *
  ep 18 | loss 0.1488 | val-Dice 0.8932
  ep 19 

In [1]:
# ══════════════════════════════════════════════════════════════════════
# VISUALISE 5-FOLD SEGMENTATION — green = ground truth, red = predicted
#   best / median / worst case per fold, plus Dice distribution
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","seg_folds"); os.makedirs(FIG,exist_ok=True)
S=512
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

def overlay(img_p, msk_p, pred_p):
    im=cv2.imread(img_p,cv2.IMREAD_GRAYSCALE)
    if im is None: return None
    im=cv2.resize(im,(S,S)); g=_clahe.apply(im)
    ov=cv2.cvtColor(g,cv2.COLOR_GRAY2RGB)
    gt=cv2.imread(msk_p,cv2.IMREAD_GRAYSCALE)
    if gt is not None:
        gm=(cv2.resize(gt,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        c,_=cv2.findContours(gm,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov,c,-1,(0,255,0),3)                 # green = truth
    pr=cv2.imread(pred_p,cv2.IMREAD_GRAYSCALE)
    if pr is not None:
        pm=(cv2.resize(pr,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        c,_=cv2.findContours(pm,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov,c,-1,(255,60,60),3)                # red = predicted
    return ov

def sheet(LES):
    d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv"))
    if "oof_dice" not in d.columns:
        print(LES+": no oof_dice column — rerun Phase 2"); return
    PM=os.path.join(D,"predmasks_"+LES)
    d["pred"]=d["img"].apply(lambda p: os.path.join(PM,os.path.basename(p).replace("_img.png","")+"_pred.png"))
    d=d[d.pred.apply(os.path.exists)].reset_index(drop=True)
    print(LES.upper()+": "+str(len(d))+" predicted masks   mean Dice "+format(d.oof_dice.mean(),".4f"))

    fig,ax=plt.subplots(5,3,figsize=(11,18))
    for k in range(5):
        q=d[d.fold==k].sort_values("oof_dice").reset_index(drop=True)
        if len(q)==0: continue
        picks=[("worst",q.iloc[0]),("median",q.iloc[len(q)//2]),("best",q.iloc[-1])]
        for c,(tag,r) in enumerate(picks):
            a=ax[k,c]; a.axis("off")
            ov=overlay(r["img"],r["msk"],r["pred"])
            if ov is None: continue
            a.imshow(ov)
            a.set_title("fold "+str(k)+" — "+tag+"\nDice "+format(r["oof_dice"],".3f")+
                        "  BI-RADS "+str(int(r["assessment"]) if pd.notna(r["assessment"]) else "?"),fontsize=9)
    plt.suptitle(LES.upper()+" segmentation — green = ground truth, red = predicted",fontsize=13)
    plt.tight_layout()
    p=os.path.join(FIG,LES+"_folds_overlay.png"); plt.savefig(p,dpi=110,bbox_inches="tight"); plt.close()
    print("  saved "+os.path.basename(p))

    # Dice distribution per fold
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].boxplot([d[d.fold==k].oof_dice.values for k in range(5)],labels=["f0","f1","f2","f3","f4"])
    axes[0].set_ylabel("Dice"); axes[0].set_title(LES+" — Dice by fold"); axes[0].grid(alpha=.25)
    axes[1].hist(d.oof_dice,bins=40,color="#1f6fb4",edgecolor="white")
    axes[1].axvline(d.oof_dice.mean(),color="red",ls="--",label="mean "+format(d.oof_dice.mean(),".3f"))
    axes[1].set_xlabel("Dice"); axes[1].set_ylabel("lesions"); axes[1].legend(); axes[1].grid(alpha=.25)
    axes[1].set_title(LES+" — Dice distribution")
    plt.tight_layout()
    p=os.path.join(FIG,LES+"_dice_dist.png"); plt.savefig(p,dpi=130,bbox_inches="tight"); plt.close()
    print("  saved "+os.path.basename(p))

    print("  per-fold: "+str([round(d[d.fold==k].oof_dice.mean(),4) for k in range(5)]))
    print("  Dice <0.5: "+str((d.oof_dice<0.5).sum())+"   <0.7: "+str((d.oof_dice<0.7).sum())+
          "   >0.9: "+str((d.oof_dice>0.9).sum()))
    lo=d.nsmallest(5,"oof_dice")
    print("  worst 5 — BI-RADS: "+str(lo.assessment.tolist())+"   coverage: "+
          str([round(100*(cv2.imread(r,cv2.IMREAD_GRAYSCALE)>127).mean(),1) for r in lo.msk]))

sheet("mass")
sheet("calc")

MASS: 1696 predicted masks   mean Dice 0.8933
  saved mass_folds_overlay.png


/tmp/ipykernel_1823/449127969.py:56: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[0].boxplot([d[d.fold==k].oof_dice.values for k in range(5)],labels=["f0","f1","f2","f3","f4"])


  saved mass_dice_dist.png
  per-fold: [np.float64(0.8873), np.float64(0.885), np.float64(0.8959), np.float64(0.9019), np.float64(0.8964)]
  Dice <0.5: 16   <0.7: 48   >0.9: 1024
  worst 5 — BI-RADS: [3, 3, 2, 5, 3]   coverage: [np.float64(18.2), np.float64(23.2), np.float64(6.5), np.float64(21.7), np.float64(14.9)]
CALC: 1866 predicted masks   mean Dice 0.8057
  saved calc_folds_overlay.png


/tmp/ipykernel_1823/449127969.py:56: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[0].boxplot([d[d.fold==k].oof_dice.values for k in range(5)],labels=["f0","f1","f2","f3","f4"])


  saved calc_dice_dist.png
  per-fold: [np.float64(0.8112), np.float64(0.8029), np.float64(0.7921), np.float64(0.8069), np.float64(0.8152)]
  Dice <0.5: 108   <0.7: 282   >0.9: 418
  worst 5 — BI-RADS: [3, 2, 2, 2, 2]   coverage: [np.float64(10.3), np.float64(1.8), np.float64(2.0), np.float64(3.7), np.float64(2.3)]


In [4]:
# ══════════════════════════════════════════════════════════════════════
# PROFILE POOR MASS SEGMENTATIONS — who are they, and why?
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv"))
d=d[d["oof_dice"].notna()].reset_index(drop=True)
d["assessment"]=pd.to_numeric(d["assessment"],errors="coerce")

cov=[];bright=[];npieces=[]
PM=os.path.join(D,"predmasks_mass")
predcov=[]
for _,r in d.iterrows():
    m=cv2.imread(str(r["msk"]),cv2.IMREAD_GRAYSCALE)
    im=cv2.imread(str(r["img"]),cv2.IMREAD_GRAYSCALE)
    b=(m>127).astype(np.uint8) if m is not None else None
    cov.append(float(b.mean()) if b is not None else np.nan)
    bright.append(float(im[b>0].mean()) if (im is not None and b is not None and b.sum()>0) else np.nan)
    npieces.append(cv2.connectedComponents(b)[0]-1 if b is not None else np.nan)
    p=os.path.join(PM,os.path.basename(str(r["img"])).replace("_img.png","")+"_pred.png")
    pm=cv2.imread(p,cv2.IMREAD_GRAYSCALE)
    predcov.append(float((pm>127).mean()) if pm is not None else np.nan)
d["mask_cov"]=cov; d["lesion_bright"]=bright; d["gt_pieces"]=npieces; d["pred_cov"]=predcov

def clean(x): return str(x).split("-")[0].replace("_"," ").lower() if pd.notna(x) else "not recorded"

for label,q in [("WORST — Dice < 0.5", d[d.oof_dice<0.5]),
                ("POOR  — Dice 0.5-0.7", d[(d.oof_dice>=0.5)&(d.oof_dice<0.7)]),
                ("GOOD  — Dice > 0.9", d[d.oof_dice>0.9])]:
    print("\n"+"="*72); print(label+"    n="+str(len(q))+
          "  ("+format(100*len(q)/len(d),".1f")+"% of "+str(len(d))+")"); print("="*72)
    if len(q)==0: continue
    br=q.assessment.value_counts().sort_index()
    base=d.assessment.value_counts().sort_index()
    print("  BI-RADS:")
    for k in sorted(set(br.index)|set(base.index)):
        n=int(br.get(k,0)); tot=int(base.get(k,0))
        if tot==0: continue
        print("    "+str(int(k))+"   "+str(n).rjust(4)+" of "+str(tot).rjust(4)+
              "   ("+format(100*n/tot,".1f").rjust(5)+"% of that category)")
    print("  shapes: ",dict(q.mass_shape.map(clean).value_counts().head(5)))
    print("  margins:",dict(q.mass_margins.map(clean).value_counts().head(5)))
    print("  subtlety mean "+format(pd.to_numeric(q.subtlety,errors='coerce').mean(),".2f")+
          "   |  GT coverage median "+format(100*q.mask_cov.median(),".1f")+"%"+
          "   |  pred coverage median "+format(100*q.pred_cov.median(),".1f")+"%")
    print("  GT mask pieces median "+format(q.gt_pieces.median(),".0f")+
          "   |  lesion brightness "+format(q.lesion_bright.mean(),".0f")+"/255")
    print("  malignant "+format(100*q.label.mean(),".0f")+"%   (dataset "+format(100*d.label.mean(),".0f")+"%)")

print("\n"+"="*72); print("DOES THE MODEL UNDER- OR OVER-SEGMENT THE BAD CASES?"); print("="*72)
bad=d[d.oof_dice<0.7]
print("  bad cases: GT coverage "+format(100*bad.mask_cov.median(),".1f")+
      "%  vs predicted "+format(100*bad.pred_cov.median(),".1f")+"%")
print("  empty predictions (pred coverage < 1%): "+str(int((bad.pred_cov<0.01).sum()))+" of "+str(len(bad)))
print("  over-predictions (pred > 2x GT):        "+str(int((bad.pred_cov>2*bad.mask_cov).sum())))

print("\n  20 worst cases:")
print("    Dice   BIRADS  GTcov  predcov  shape / margins")
for _,r in d.nsmallest(20,"oof_dice").iterrows():
    print("    "+format(r.oof_dice,".3f")+"   "+str(int(r.assessment) if pd.notna(r.assessment) else "?").rjust(3)+
          "    "+format(100*r.mask_cov,".1f").rjust(5)+"%  "+format(100*r.pred_cov,".1f").rjust(5)+"%   "+
          clean(r.mass_shape)+" / "+clean(r.mass_margins))
d.to_csv(os.path.join(D,"mass_seg_quality.csv"),index=False)
print("\n  saved mass_seg_quality.csv")


WORST — Dice < 0.5    n=16  (0.9% of 1696)
  BI-RADS:
    0      2 of  162   (  1.2% of that category)
    1      0 of    3   (  0.0% of that category)
    2      1 of   91   (  1.1% of that category)
    3      4 of  364   (  1.1% of that category)
    4      4 of  702   (  0.6% of that category)
    5      5 of  374   (  1.3% of that category)
  shapes:  {'round': np.int64(5), 'oval': np.int64(3), 'irregular': np.int64(3), 'lobulated': np.int64(2), 'lymph node': np.int64(1)}
  margins: {'circumscribed': np.int64(7), 'ill defined': np.int64(3), 'spiculated': np.int64(2), 'microlobulated': np.int64(2), 'obscured': np.int64(2)}
  subtlety mean 4.12   |  GT coverage median 18.3%   |  pred coverage median 20.4%
  GT mask pieces median 1   |  lesion brightness 154/255
  malignant 50%   (dataset 46%)

POOR  — Dice 0.5-0.7    n=32  (1.9% of 1696)
  BI-RADS:
    0      3 of  162   (  1.9% of that category)
    1      0 of    3   (  0.0% of that category)
    2      7 of   91   (  7.7% of tha

In [5]:
# ══════════════════════════════════════════════════════════════════════
# SEGMENTATION TTA + THRESHOLD TUNING (no retraining)
#   TTA: average predictions over 4 flips/rotations - stabilises localisation
#   threshold chosen on each fold's VALIDATION set, applied to its test set
# ══════════════════════════════════════════════════════════════════════
import os, time, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda")
IMG=256; BATCH=16
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

# --- same architecture as Phase 2 (DS-Attn-UNet + ASPP) ---
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class ASPP(nn.Module):
    def __init__(s,i,o):
        super().__init__()
        s.b0=nn.Sequential(nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b1=nn.Sequential(nn.Conv2d(i,o,3,padding=6,dilation=6),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b2=nn.Sequential(nn.Conv2d(i,o,3,padding=12,dilation=12),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b3=nn.Sequential(nn.Conv2d(i,o,3,padding=18,dilation=18),nn.BatchNorm2d(o),nn.ReLU(True))
        s.gp=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.proj=nn.Sequential(nn.Conv2d(o*5,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(s,x):
        g=F.interpolate(s.gp(x),size=x.shape[2:],mode="bilinear",align_corners=False)
        return s.proj(torch.cat([s.b0(x),s.b1(x),s.b2(x),s.b3(x),g],1))
class DSAttnUNet(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=ASPP(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out=nn.Conv2d(b,2,1); s.ds2=nn.Conv2d(b*2,2,1); s.ds3=nn.Conv2d(b*4,2,1); s.ds4=nn.Conv2d(b*8,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        return s.out(d1)

@torch.no_grad()
def prob_tta(net,imgs):
    """average softmax over identity, hflip, vflip, rot180"""
    x=torch.from_numpy(np.stack(imgs)[:,None]).float().to(DEV)
    acc=None
    for t in range(4):
        xt = x if t==0 else (torch.flip(x,[3]) if t==1 else (torch.flip(x,[2]) if t==2 else torch.flip(x,[2,3])))
        with torch.amp.autocast(device_type="cuda"): o=net(xt)
        p=torch.softmax(o.float(),1)[:,1:2]
        p = p if t==0 else (torch.flip(p,[3]) if t==1 else (torch.flip(p,[2]) if t==2 else torch.flip(p,[2,3])))
        acc = p if acc is None else acc+p
    return (acc/4)[:,0].cpu().numpy()

def dice(m,g):
    tp=(m*g).sum(); fp=(m*(1-g)).sum(); fn=((1-m)*g).sum()
    return (2*tp+1)/(2*tp+fp+fn+1)

def run(LES):
    d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv")).reset_index(drop=True)
    OUT=os.path.join(D,"predmasks_"+LES); os.makedirs(OUT,exist_ok=True)
    CACHE={}
    for _,r in d.iterrows():
        k=r["img"]
        if k in CACHE: continue
        im=cv2.imread(k,cv2.IMREAD_GRAYSCALE); im=cv2.resize(im,(IMG,IMG))
        mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        gt=(cv2.resize(mk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32)
        CACHE[k]=(_clahe.apply(im).astype(np.float32)/255., gt)

    base=[]; tta=[]; tuned=[]
    for k in range(5):
        ck=os.path.join(D,"seg_dsaspp_"+LES+"_fold"+str(k)+".pth")
        if not os.path.exists(ck): print("missing "+ck); return
        net=DSAttnUNet().to(DEV); net.load_state_dict(torch.load(ck,map_location="cpu")); net.eval()
        role=d["role_f"+str(k)]
        vi=d.index[role=="val"].values; ti=d.index[role=="test"].values

        def probs_for(idx):
            P={}
            for i0 in range(0,len(idx),BATCH):
                b=idx[i0:i0+BATCH]
                pr=prob_tta(net,[CACHE[d.iloc[j]["img"]][0] for j in b])
                for q,j in enumerate(b): P[j]=pr[q]
            return P
        Pv=probs_for(vi); Pt=probs_for(ti)

        # threshold from validation only
        best=(0,0.5)
        for T in np.arange(0.20,0.81,0.02):
            s=np.mean([dice((Pv[j]>T).astype(np.float32),CACHE[d.iloc[j]["img"]][1]) for j in vi])
            if s>best[0]: best=(s,T)
        T=best[1]

        for j in ti:
            gt=CACHE[d.iloc[j]["img"]][1]
            base.append(d.iloc[j]["oof_dice"])
            tta.append(dice((Pt[j]>0.5).astype(np.float32),gt))
            m=(Pt[j]>T).astype(np.float32); tuned.append(dice(m,gt))
            cv2.imwrite(os.path.join(OUT,os.path.basename(d.iloc[j]["img"]).replace("_img.png","")+"_pred.png"),
                        cv2.resize((m*255).astype(np.uint8),(512,512),interpolation=cv2.INTER_NEAREST))
        print("  fold "+str(k)+"  thr "+format(T,".2f")+
              "  base "+format(np.mean([d.iloc[j]['oof_dice'] for j in ti]),".4f")+
              "  +TTA "+format(np.mean([dice((Pt[j]>0.5).astype(np.float32),CACHE[d.iloc[j]['img']][1]) for j in ti]),".4f")+
              "  +thr "+format(np.mean([dice((Pt[j]>T).astype(np.float32),CACHE[d.iloc[j]['img']][1]) for j in ti]),".4f"))

    print("\n"+LES.upper())
    print("  original      "+format(np.mean(base),".4f"))
    print("  + TTA         "+format(np.mean(tta),".4f")+"   ("+format(np.mean(tta)-np.mean(base),"+.4f")+")")
    print("  + TTA + thr   "+format(np.mean(tuned),".4f")+"   ("+format(np.mean(tuned)-np.mean(base),"+.4f")+")")
    print("  Dice<0.7: "+str(int((np.array(base)<0.7).sum()))+" -> "+str(int((np.array(tuned)<0.7).sum())))
    d.loc[:,"oof_dice_tta"]=np.nan
    d.to_csv(os.path.join(D,"unified_folds_"+LES+".csv"),index=False)

run("mass"); run("calc")

  fold 0  thr 0.28  base 0.8873  +TTA 0.8901  +thr 0.8952
  fold 1  thr 0.52  base 0.8850  +TTA 0.8927  +thr 0.8930
  fold 2  thr 0.32  base 0.8959  +TTA 0.9010  +thr 0.9037
  fold 3  thr 0.34  base 0.9019  +TTA 0.9054  +thr 0.9074
  fold 4  thr 0.30  base 0.8964  +TTA 0.9000  +thr 0.8999

MASS
  original      0.8933
  + TTA         0.8978   (+0.0045)
  + TTA + thr   0.8998   (+0.0066)
  Dice<0.7: 48 -> 41
  fold 0  thr 0.46  base 0.8112  +TTA 0.8117  +thr 0.8119
  fold 1  thr 0.22  base 0.8029  +TTA 0.8051  +thr 0.8121
  fold 2  thr 0.30  base 0.7921  +TTA 0.7947  +thr 0.7967
  fold 3  thr 0.52  base 0.8069  +TTA 0.8070  +thr 0.8065
  fold 4  thr 0.26  base 0.8152  +TTA 0.8162  +thr 0.8180

CALC
  original      0.8057
  + TTA         0.8069   (+0.0013)
  + TTA + thr   0.8090   (+0.0034)
  Dice<0.7: 282 -> 290


In [6]:
# ══════════════════════════════════════════════════════════════════════
# VISUALISE TTA+threshold EFFECT — before vs after, per lesion type
#   recomputes Dice from the saved (tuned) masks and compares to original
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","seg_tta"); os.makedirs(FIG,exist_ok=True)
S=512
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

def dice_np(m,g):
    tp=(m&g).sum(); fp=(m&~g).sum(); fn=((~m)&g).sum()
    return (2*tp+1)/(2*tp+fp+fn+1)

def overlay(img_p,msk_p,pred_p):
    im=cv2.imread(img_p,cv2.IMREAD_GRAYSCALE)
    if im is None: return None
    ov=cv2.cvtColor(_clahe.apply(cv2.resize(im,(S,S))),cv2.COLOR_GRAY2RGB)
    gt=cv2.imread(msk_p,cv2.IMREAD_GRAYSCALE)
    if gt is not None:
        g=(cv2.resize(gt,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        c,_=cv2.findContours(g,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(ov,c,-1,(0,255,0),3)
    pr=cv2.imread(pred_p,cv2.IMREAD_GRAYSCALE)
    if pr is not None:
        p=(cv2.resize(pr,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        c,_=cv2.findContours(p,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(ov,c,-1,(255,60,60),3)
    return ov

def report(LES):
    d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv"))
    d=d[d["oof_dice"].notna()].reset_index(drop=True)
    PM=os.path.join(D,"predmasks_"+LES)
    d["pred"]=d["img"].apply(lambda p: os.path.join(PM,os.path.basename(p).replace("_img.png","")+"_pred.png"))
    d=d[d["pred"].apply(os.path.exists)].reset_index(drop=True)

    new=[]
    for _,r in d.iterrows():
        g=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE); p=cv2.imread(r["pred"],cv2.IMREAD_GRAYSCALE)
        if g is None or p is None: new.append(np.nan); continue
        gg=(cv2.resize(g,(256,256),interpolation=cv2.INTER_NEAREST)>127)
        pp=(cv2.resize(p,(256,256),interpolation=cv2.INTER_NEAREST)>127)
        new.append(dice_np(pp,gg))
    d["dice_new"]=new
    d["delta"]=d["dice_new"]-d["oof_dice"]
    d.to_csv(os.path.join(D,"seg_tta_"+LES+".csv"),index=False)

    print("\n"+"="*62); print(LES.upper()+"   n="+str(len(d))); print("="*62)
    print("  before "+format(d.oof_dice.mean(),".4f")+"   after "+format(d.dice_new.mean(),".4f")+
          "   delta "+format(d.delta.mean(),"+.4f"))
    print("  improved: "+str(int((d.delta>0.01).sum()))+"   unchanged: "+
          str(int((d.delta.abs()<=0.01).sum()))+"   degraded: "+str(int((d.delta<-0.01).sum())))
    print("  Dice<0.7:  "+str(int((d.oof_dice<0.7).sum()))+" -> "+str(int((d.dice_new<0.7).sum())))
    print("  Dice>0.9:  "+str(int((d.oof_dice>0.9).sum()))+" -> "+str(int((d.dice_new>0.9).sum())))

    # scatter + histogram
    fig,ax=plt.subplots(1,3,figsize=(16,4.6))
    ax[0].scatter(d.oof_dice,d.dice_new,s=6,alpha=.35,color="#1f6fb4")
    ax[0].plot([0,1],[0,1],"k--",lw=1)
    ax[0].set_xlabel("Dice before"); ax[0].set_ylabel("Dice after TTA+thr")
    ax[0].set_title(LES+" — per-lesion change"); ax[0].grid(alpha=.25)
    ax[1].hist(d.delta,bins=60,color="#c26a3d",edgecolor="white")
    ax[1].axvline(0,color="k",lw=1); ax[1].axvline(d.delta.mean(),color="red",ls="--",
        label="mean "+format(d.delta.mean(),"+.4f"))
    ax[1].set_xlabel("Dice change"); ax[1].set_ylabel("lesions"); ax[1].legend(); ax[1].grid(alpha=.25)
    ax[1].set_title("improvement distribution")
    ax[2].hist([d.oof_dice,d.dice_new],bins=40,label=["before","after"],color=["#b0aea6","#1f6fb4"])
    ax[2].axvline(0.7,color="red",ls=":",lw=1.5,label="Dice 0.7")
    ax[2].set_xlabel("Dice"); ax[2].legend(); ax[2].grid(alpha=.25); ax[2].set_title("distribution shift")
    plt.tight_layout(); plt.savefig(os.path.join(FIG,LES+"_tta_effect.png"),dpi=130,bbox_inches="tight"); plt.close()

    # visual: most improved / still worst
    for tag,sel in [("most_improved",d.nlargest(6,"delta")),("still_worst",d.nsmallest(6,"dice_new"))]:
        fig,axx=plt.subplots(2,3,figsize=(12,8)); axx=axx.ravel()
        for i,(_,r) in enumerate(sel.iterrows()):
            a=axx[i]; a.axis("off")
            ov=overlay(r["img"],r["msk"],r["pred"])
            if ov is None: continue
            a.imshow(ov)
            a.set_title("Dice "+format(r["oof_dice"],".2f")+" -> "+format(r["dice_new"],".2f")+
                        "\nBI-RADS "+str(int(r["assessment"]) if pd.notna(r["assessment"]) else "?")+
                        "  "+str(r.get("mass_shape","")).split("-")[0][:14].lower(),fontsize=8)
        plt.suptitle(LES.upper()+" — "+tag.replace("_"," ")+"   (green = truth, red = predicted)",fontsize=12)
        plt.tight_layout(); plt.savefig(os.path.join(FIG,LES+"_"+tag+".png"),dpi=110,bbox_inches="tight"); plt.close()
    print("  figures -> "+FIG)

report("mass"); report("calc")


MASS   n=1696
  before 0.8933   after 0.8998   delta +0.0066
  improved: 601   unchanged: 881   degraded: 214
  Dice<0.7:  48 -> 41
  Dice>0.9:  1024 -> 1121
  figures -> /root/autodl-tmp/CBIS/figures/seg_tta

CALC   n=1866
  before 0.8057   after 0.8090   delta +0.0034
  improved: 514   unchanged: 920   degraded: 432
  Dice<0.7:  282 -> 290
  Dice>0.9:  418 -> 508
  figures -> /root/autodl-tmp/CBIS/figures/seg_tta


In [8]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 3 — SEGMENTATION-GUIDED CLASSIFICATION on unified folds
#
#   Same folds as Phase 2. Two conditions trained per fold:
#     (A) guided by REFERENCE masks   (what you had before)
#     (B) guided by PREDICTED masks   (from Phase 2, patient-blind)
#   Everything else identical: DenseNet-121, 512px, guided attention
#   w = 1 + 2*mask, multi-task aux heads, focal loss, 8x aug, 4-way TTA
#
#   Set LES = "mass" or "calc"
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda")

LES="mass"                                   # "mass" or "calc"
AUXC = ["subtlety","mass_shape","mass_margins"] if LES=="mass" else ["subtlety","calc_type","calc_dist"]
S=512; BATCH=12; MULT=8; EPOCHS=20; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; AUX_W=0.3
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv")).reset_index(drop=True)
d["label"]=d["label"].astype(int); d["assessment"]=pd.to_numeric(d["assessment"],errors="coerce")
PM=os.path.join(D,"predmasks_"+LES)
d["pred"]=d["img"].apply(lambda p: os.path.join(PM,os.path.basename(p).replace("_img.png","")+"_pred.png"))
have=d["pred"].apply(os.path.exists)
print(LES.upper()+"   n="+str(len(d))+"   predicted masks available: "+str(int(have.sum())))
assert have.all(), "missing predicted masks - rerun Phase 2"

CACHE={}; t0=time.time()
for _,r in d.iterrows():
    k=r["img"]
    if k in CACHE: continue
    im=cv2.imread(k,cv2.IMREAD_GRAYSCALE); im=cv2.resize(im,(S,S))
    gt=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    gm=(cv2.resize(gt,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if gt is not None else np.zeros((S,S),np.float32)
    pd_=cv2.imread(r["pred"],cv2.IMREAD_GRAYSCALE)
    pmm=(cv2.resize(pd_,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if pd_ is not None else np.zeros((S,S),np.float32)
    CACHE[k]=(_clahe.apply(im),gm,pmm)
print("cached "+str(len(CACHE))+" in "+format(time.time()-t0,".0f")+"s")

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux={}; meta={}
for c in AUXC:
    if c not in d.columns or d[c].notna().sum()==0: continue
    if c=="subtlety":
        v=pd.to_numeric(d[c],errors="coerce").where(lambda z:(z>=1)&(z<=5))
        codes=(v-1).fillna(-1).astype(int); n=int(v.max()) if v.notna().any() else 0
    else:
        pr=d[c].map(primary); keep=pr.value_counts().head(6).index.tolist()
        pr=pr.where(pr.isin(keep),"OTHER")
        cats=sorted([k for k in pr.unique() if k!="UNK"]); mp={k:i for i,k in enumerate(cats)}
        codes=pr.map(lambda z:mp.get(z,-1)).astype(int); n=len(cats)
    if n>1: aux[c]=codes.values; meta[c]=n
AK=sorted(aux.keys()); print("aux heads:",meta)

class DS(Dataset):
    def __init__(s,idx,use_pred,aug,tta=0):
        s.idx=np.array(idx); s.up=use_pred; s.aug=aug; s.m=MULT if aug else 1; s.tta=tta
    def __len__(s): return len(s.idx)*s.m
    def __getitem__(s,i):
        j=s.idx[i%len(s.idx)]; v=i//len(s.idx); r=d.iloc[j]
        img,gm,pm=CACHE[r["img"]]
        mask=(pm if s.up else gm).copy(); img=img.copy()
        if s.aug and v>0:
            if   v==1: img=np.fliplr(img); mask=np.fliplr(mask)
            elif v==2: img=np.flipud(img); mask=np.flipud(mask)
            elif v==3: img=np.rot90(img,1); mask=np.rot90(mask,1)
            elif v==4: img=np.rot90(img,2); mask=np.rot90(mask,2)
            elif v==5: img=np.rot90(img,3); mask=np.rot90(mask,3)
            elif v==6:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-20,20),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT)
                mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            elif v==7: img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        if s.tta==1: img=np.fliplr(img); mask=np.fliplr(mask)
        elif s.tta==2: img=np.flipud(img); mask=np.flipud(mask)
        elif s.tta==3: img=np.rot90(img,2); mask=np.rot90(mask,2)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        av=np.array([aux[c][j] for c in AK],dtype=np.int64) if AK else np.zeros(0,np.int64)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(mask))[None],
                torch.tensor(int(r["label"])), torch.from_numpy(av))

class Net(nn.Module):
    def __init__(s,meta):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        s.keys=sorted(meta.keys())
        s.aux=nn.ModuleList([nn.Sequential(nn.Linear(1024,128),nn.ReLU(),nn.Dropout(0.3),
                                           nn.Linear(128,meta[k])) for k in s.keys])
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+2.0*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        return s.head(g),[h(g) for h in s.aux]

y=d["label"].values
OOF={"ref":np.zeros(len(d)),"pred":np.zeros(len(d))}

for cond,use_pred in [("ref",False),("pred",True)]:
    print("\n"+"#"*70)
    print("#  CONDITION: guided by "+("PREDICTED" if use_pred else "REFERENCE")+" masks")
    print("#"*70)
    for k in range(5):
        role=d["role_f"+str(k)]
        tr=np.where(role=="train")[0]; va=np.where(role=="val")[0]; te=np.where(role=="test")[0]
        assert not (set(d.patient_id[tr])&set(d.patient_id[te])), "LEAK"
        t0=time.time(); torch.manual_seed(k); np.random.seed(k)
        n0=float((y[tr]==0).sum()); n1=float((y[tr]==1).sum())
        al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
        def focal(lo,t):
            ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce)
            return ((1-pt)**GAMMA*ce).mean()
        net=Net(meta).to(DEV).to(memory_format=torch.channels_last)
        for p_ in net.b.parameters(): p_.requires_grad=False
        sc=torch.amp.GradScaler()
        opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
        tl=DataLoader(DS(tr,use_pred,True),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
        @torch.no_grad()
        def col(idx,tta=True):
            net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None
            for t in reps:
                ld=DataLoader(DS(idx,use_pred,False,tta=t),batch_size=20,shuffle=False,num_workers=0)
                ps=[]
                for x,m,_,_ in ld:
                    x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                    with torch.amp.autocast(device_type="cuda"): o,_=net(x,m)
                    ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
                ps=np.array(ps); tot=ps if tot is None else tot+ps
            return tot/len(reps)
        best=0;bs=None;ni=0
        for ep in range(1,EPOCHS+1):
            if ep==FREEZE+1:
                for p_ in net.b.parameters(): p_.requires_grad=True
                opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
            net.train()
            if ep<=FREEZE: net.b.eval()
            for x,m,t2,a in tl:
                x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                t2=t2.to(DEV); a=a.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"):
                    o,ax=net(x,m); L=focal(o,t2)
                    if len(ax):
                        la=sum(F.cross_entropy(g.float(),a[:,h],ignore_index=-1) for h,g in enumerate(ax))/len(ax)
                        L=L+AUX_W*la
                if not torch.isfinite(L): continue
                sc.scale(L).backward(); sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
            pv=col(va,tta=False); au=roc_auc_score(y[va],pv) if len(set(y[va]))>1 else 0
            if au>best: best=au; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0
            else: ni+=1
            if ni>=5: break
        net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
        OOF[cond][te]=col(te)
        print("  fold "+str(k)+"  AUC "+format(roc_auc_score(y[te],OOF[cond][te]),".4f")+
              "   ("+format(time.time()-t0,".0f")+"s)")

d["prob_ref"]=OOF["ref"]; d["prob_pred"]=OOF["pred"]; d["true"]=y
d.to_csv(os.path.join(D,"phase3_"+LES+".csv"),index=False)

print("\n"+"="*72)
print("PHASE 3 — "+LES.upper()+"   reference vs predicted masks")
print("="*72)
for cond,tag in [("ref","REFERENCE masks"),("pred","PREDICTED masks (end-to-end)")]:
    p=OOF[cond]
    thr=max([(balanced_accuracy_score(y,(p>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    L=d.groupby("lesion_key").agg(yy=("true","max"),pp=(("prob_"+cond),"mean")).reset_index()
    thrL=max([(balanced_accuracy_score(L.yy,(L.pp>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    pr=(L.pp>thrL).astype(int); tn,fp,fn,tp=confusion_matrix(L.yy,pr,labels=[0,1]).ravel()
    print("\n  "+tag)
    print("    per-image  AUC "+format(roc_auc_score(y,p),".4f")+
          "  acc "+format(100*accuracy_score(y,(p>thr).astype(int)),".1f")+"%")
    print("    per-lesion AUC "+format(roc_auc_score(L.yy,L.pp),".4f")+
          "  acc "+format(100*accuracy_score(L.yy,pr),".1f")+"%"+
          "  sens "+format(tp/max(tp+fn,1),".3f")+"  spec "+format(tn/max(tn+fp,1),".3f")+
          "  FP "+str(fp)+" FN "+str(fn))
Lr=d.groupby("lesion_key").agg(yy=("true","max"),r=("prob_ref","mean"),q=("prob_pred","mean")).reset_index()
print("\n  cost of using predicted masks: "+
      format(roc_auc_score(Lr.yy,Lr.q)-roc_auc_score(Lr.yy,Lr.r),"+.4f")+" AUC (per-lesion)")
print("  Tsochatzidis 2021 measured 0.862 -> 0.860 for the same substitution")
print("="*72)

MASS   n=1696   predicted masks available: 1696
cached 1696 in 11s
aux heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

######################################################################
#  CONDITION: guided by REFERENCE masks
######################################################################
  fold 0  AUC 0.7847   (2128s)
  fold 1  AUC 0.8736   (1276s)
  fold 2  AUC 0.7951   (2057s)
  fold 3  AUC 0.8497   (1706s)
  fold 4  AUC 0.8522   (1922s)

######################################################################
#  CONDITION: guided by PREDICTED masks
######################################################################
  fold 0  AUC 0.7871   (1272s)
  fold 1  AUC 0.8585   (1047s)
  fold 2  AUC 0.7840   (1137s)
  fold 3  AUC 0.7213   (492s)
  fold 4  AUC 0.8522   (1807s)

PHASE 3 — MASS   reference vs predicted masks

  REFERENCE masks
    per-image  AUC 0.8247  acc 76.4%
    per-lesion AUC 0.8441  acc 78.2%  sens 0.784  spec 0.781  FP 119 FN 100

  PREDICTED mas

In [1]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 3 — SEGMENTATION-GUIDED CLASSIFICATION on unified folds
#
#   Same folds as Phase 2. Two conditions trained per fold:
#     (A) guided by REFERENCE masks   (what you had before)
#     (B) guided by PREDICTED masks   (from Phase 2, patient-blind)
#   Everything else identical: DenseNet-121, 512px, guided attention
#   w = 1 + 2*mask, multi-task aux heads, focal loss, 8x aug, 4-way TTA
#
#   Set LES = "mass" or "calc"
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda")

LES="calc"                                   # "mass" or "calc"
AUXC = ["subtlety","mass_shape","mass_margins"] if LES=="mass" else ["subtlety","calc_type","calc_dist"]
S=512; BATCH=12; MULT=8; EPOCHS=20; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; AUX_W=0.3
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv")).reset_index(drop=True)
d["label"]=d["label"].astype(int); d["assessment"]=pd.to_numeric(d["assessment"],errors="coerce")
PM=os.path.join(D,"predmasks_"+LES)
d["pred"]=d["img"].apply(lambda p: os.path.join(PM,os.path.basename(p).replace("_img.png","")+"_pred.png"))
have=d["pred"].apply(os.path.exists)
print(LES.upper()+"   n="+str(len(d))+"   predicted masks available: "+str(int(have.sum())))
assert have.all(), "missing predicted masks - rerun Phase 2"

CACHE={}; t0=time.time()
for _,r in d.iterrows():
    k=r["img"]
    if k in CACHE: continue
    im=cv2.imread(k,cv2.IMREAD_GRAYSCALE); im=cv2.resize(im,(S,S))
    gt=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    gm=(cv2.resize(gt,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if gt is not None else np.zeros((S,S),np.float32)
    pd_=cv2.imread(r["pred"],cv2.IMREAD_GRAYSCALE)
    pmm=(cv2.resize(pd_,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if pd_ is not None else np.zeros((S,S),np.float32)
    CACHE[k]=(_clahe.apply(im),gm,pmm)
print("cached "+str(len(CACHE))+" in "+format(time.time()-t0,".0f")+"s")

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux={}; meta={}
for c in AUXC:
    if c not in d.columns or d[c].notna().sum()==0: continue
    if c=="subtlety":
        v=pd.to_numeric(d[c],errors="coerce").where(lambda z:(z>=1)&(z<=5))
        codes=(v-1).fillna(-1).astype(int); n=int(v.max()) if v.notna().any() else 0
    else:
        pr=d[c].map(primary); keep=pr.value_counts().head(6).index.tolist()
        pr=pr.where(pr.isin(keep),"OTHER")
        cats=sorted([k for k in pr.unique() if k!="UNK"]); mp={k:i for i,k in enumerate(cats)}
        codes=pr.map(lambda z:mp.get(z,-1)).astype(int); n=len(cats)
    if n>1: aux[c]=codes.values; meta[c]=n
AK=sorted(aux.keys()); print("aux heads:",meta)

class DS(Dataset):
    def __init__(s,idx,use_pred,aug,tta=0):
        s.idx=np.array(idx); s.up=use_pred; s.aug=aug; s.m=MULT if aug else 1; s.tta=tta
    def __len__(s): return len(s.idx)*s.m
    def __getitem__(s,i):
        j=s.idx[i%len(s.idx)]; v=i//len(s.idx); r=d.iloc[j]
        img,gm,pm=CACHE[r["img"]]
        mask=(pm if s.up else gm).copy(); img=img.copy()
        if s.aug and v>0:
            if   v==1: img=np.fliplr(img); mask=np.fliplr(mask)
            elif v==2: img=np.flipud(img); mask=np.flipud(mask)
            elif v==3: img=np.rot90(img,1); mask=np.rot90(mask,1)
            elif v==4: img=np.rot90(img,2); mask=np.rot90(mask,2)
            elif v==5: img=np.rot90(img,3); mask=np.rot90(mask,3)
            elif v==6:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-20,20),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT)
                mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            elif v==7: img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        if s.tta==1: img=np.fliplr(img); mask=np.fliplr(mask)
        elif s.tta==2: img=np.flipud(img); mask=np.flipud(mask)
        elif s.tta==3: img=np.rot90(img,2); mask=np.rot90(mask,2)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        av=np.array([aux[c][j] for c in AK],dtype=np.int64) if AK else np.zeros(0,np.int64)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(mask))[None],
                torch.tensor(int(r["label"])), torch.from_numpy(av))

class Net(nn.Module):
    def __init__(s,meta):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        s.keys=sorted(meta.keys())
        s.aux=nn.ModuleList([nn.Sequential(nn.Linear(1024,128),nn.ReLU(),nn.Dropout(0.3),
                                           nn.Linear(128,meta[k])) for k in s.keys])
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+2.0*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        return s.head(g),[h(g) for h in s.aux]

y=d["label"].values
OOF={"ref":np.zeros(len(d)),"pred":np.zeros(len(d))}

for cond,use_pred in [("ref",False),("pred",True)]:
    print("\n"+"#"*70)
    print("#  CONDITION: guided by "+("PREDICTED" if use_pred else "REFERENCE")+" masks")
    print("#"*70)
    for k in range(5):
        role=d["role_f"+str(k)]
        tr=np.where(role=="train")[0]; va=np.where(role=="val")[0]; te=np.where(role=="test")[0]
        assert not (set(d.patient_id[tr])&set(d.patient_id[te])), "LEAK"
        t0=time.time(); torch.manual_seed(k); np.random.seed(k)
        n0=float((y[tr]==0).sum()); n1=float((y[tr]==1).sum())
        al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
        def focal(lo,t):
            ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce)
            return ((1-pt)**GAMMA*ce).mean()
        net=Net(meta).to(DEV).to(memory_format=torch.channels_last)
        for p_ in net.b.parameters(): p_.requires_grad=False
        sc=torch.amp.GradScaler()
        opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
        tl=DataLoader(DS(tr,use_pred,True),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
        @torch.no_grad()
        def col(idx,tta=True):
            net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None
            for t in reps:
                ld=DataLoader(DS(idx,use_pred,False,tta=t),batch_size=20,shuffle=False,num_workers=0)
                ps=[]
                for x,m,_,_ in ld:
                    x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                    with torch.amp.autocast(device_type="cuda"): o,_=net(x,m)
                    ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
                ps=np.array(ps); tot=ps if tot is None else tot+ps
            return tot/len(reps)
        best=0;bs=None;ni=0
        for ep in range(1,EPOCHS+1):
            if ep==FREEZE+1:
                for p_ in net.b.parameters(): p_.requires_grad=True
                opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
            net.train()
            if ep<=FREEZE: net.b.eval()
            for x,m,t2,a in tl:
                x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                t2=t2.to(DEV); a=a.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"):
                    o,ax=net(x,m); L=focal(o,t2)
                    if len(ax):
                        la=sum(F.cross_entropy(g.float(),a[:,h],ignore_index=-1) for h,g in enumerate(ax))/len(ax)
                        L=L+AUX_W*la
                if not torch.isfinite(L): continue
                sc.scale(L).backward(); sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
            pv=col(va,tta=False); au=roc_auc_score(y[va],pv) if len(set(y[va]))>1 else 0
            if au>best: best=au; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0
            else: ni+=1
            if ni>=5: break
        net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
        OOF[cond][te]=col(te)
        print("  fold "+str(k)+"  AUC "+format(roc_auc_score(y[te],OOF[cond][te]),".4f")+
              "   ("+format(time.time()-t0,".0f")+"s)")

d["prob_ref"]=OOF["ref"]; d["prob_pred"]=OOF["pred"]; d["true"]=y
d.to_csv(os.path.join(D,"phase3_"+LES+".csv"),index=False)

print("\n"+"="*72)
print("PHASE 3 — "+LES.upper()+"   reference vs predicted masks")
print("="*72)
for cond,tag in [("ref","REFERENCE masks"),("pred","PREDICTED masks (end-to-end)")]:
    p=OOF[cond]
    thr=max([(balanced_accuracy_score(y,(p>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    L=d.groupby("lesion_key").agg(yy=("true","max"),pp=(("prob_"+cond),"mean")).reset_index()
    thrL=max([(balanced_accuracy_score(L.yy,(L.pp>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    pr=(L.pp>thrL).astype(int); tn,fp,fn,tp=confusion_matrix(L.yy,pr,labels=[0,1]).ravel()
    print("\n  "+tag)
    print("    per-image  AUC "+format(roc_auc_score(y,p),".4f")+
          "  acc "+format(100*accuracy_score(y,(p>thr).astype(int)),".1f")+"%")
    print("    per-lesion AUC "+format(roc_auc_score(L.yy,L.pp),".4f")+
          "  acc "+format(100*accuracy_score(L.yy,pr),".1f")+"%"+
          "  sens "+format(tp/max(tp+fn,1),".3f")+"  spec "+format(tn/max(tn+fp,1),".3f")+
          "  FP "+str(fp)+" FN "+str(fn))
Lr=d.groupby("lesion_key").agg(yy=("true","max"),r=("prob_ref","mean"),q=("prob_pred","mean")).reset_index()
print("\n  cost of using predicted masks: "+
      format(roc_auc_score(Lr.yy,Lr.q)-roc_auc_score(Lr.yy,Lr.r),"+.4f")+" AUC (per-lesion)")
print("  Tsochatzidis 2021 measured 0.862 -> 0.860 for the same substitution")
print("="*72)

CALC   n=1866   predicted masks available: 1866
cached 1866 in 13s
aux heads: {'subtlety': 5, 'calc_type': 7, 'calc_dist': 5}

######################################################################
#  CONDITION: guided by REFERENCE masks
######################################################################
  fold 0  AUC 0.8320   (1919s)
  fold 1  AUC 0.7805   (896s)
  fold 2  AUC 0.8037   (2109s)
  fold 3  AUC 0.7779   (703s)
  fold 4  AUC 0.8336   (1779s)

######################################################################
#  CONDITION: guided by PREDICTED masks
######################################################################
  fold 0  AUC 0.8299   (2213s)
  fold 1  AUC 0.7682   (699s)
  fold 2  AUC 0.7601   (721s)
  fold 3  AUC 0.7596   (719s)
  fold 4  AUC 0.8218   (1435s)

PHASE 3 — CALC   reference vs predicted masks

  REFERENCE masks
    per-image  AUC 0.7984  acc 70.8%
    per-lesion AUC 0.7963  acc 72.4%  sens 0.719  spec 0.726  FP 181 FN 107

  PREDICTED masks (end-

In [9]:
# ══════════════════════════════════════════════════════════════════════
# FOLD 3 RE-RUN — predicted-mask condition only
#   Justification: 492s runtime vs 1047-1807s for other folds = stalled run
#   Protocol fixed BEFORE seeing results: 3 seeds, report MEDIAN not best
#   Requires the Phase 3 cell to have been run in this session
#   (uses d, CACHE, DS, Net, meta, aux, AK, y, OOF from it)
# ══════════════════════════════════════════════════════════════════════
import numpy as np, torch, torch.nn.functional as F, time
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

K=3                      # fold to re-run
SEEDS=[103,203,303]
PATIENCE=8               # was 5 - gives a stalled run more chance to recover

role=d["role_f"+str(K)]
tr=np.where(role=="train")[0]; va=np.where(role=="val")[0]; te=np.where(role=="test")[0]
assert not (set(d.patient_id[tr])&set(d.patient_id[te])), "LEAK"
print("fold "+str(K)+"  train "+str(len(tr))+"  val "+str(len(va))+"  test "+str(len(te)))
print("original predicted-mask AUC: 0.7213  (492s)")
print("reference-mask AUC:          0.8497\n")

n0=float((y[tr]==0).sum()); n1=float((y[tr]==1).sum())
al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
def focal(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce)
    return ((1-pt)**GAMMA*ce).mean()

runs=[]
for sd in SEEDS:
    t0=time.time(); torch.manual_seed(sd); np.random.seed(sd)
    net=Net(meta).to(DEV).to(memory_format=torch.channels_last)
    for p_ in net.b.parameters(): p_.requires_grad=False
    sc=torch.amp.GradScaler()
    opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
    tl=DataLoader(DS(tr,True,True),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)

    @torch.no_grad()
    def col(idx,tta=True):
        net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None
        for t in reps:
            ld=DataLoader(DS(idx,True,False,tta=t),batch_size=20,shuffle=False,num_workers=0); ps=[]
            for x,m,_,_ in ld:
                x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                with torch.amp.autocast(device_type="cuda"): o,_=net(x,m)
                ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
            ps=np.array(ps); tot=ps if tot is None else tot+ps
        return tot/len(reps)

    best=0; bs=None; ni=0; ep_best=0
    for ep in range(1,EPOCHS+1):
        if ep==FREEZE+1:
            for p_ in net.b.parameters(): p_.requires_grad=True
            opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
        net.train()
        if ep<=FREEZE: net.b.eval()
        for x,m,t2,a in tl:
            x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
            t2=t2.to(DEV); a=a.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o,ax=net(x,m); L=focal(o,t2)
                if len(ax):
                    L=L+AUX_W*sum(F.cross_entropy(g.float(),a[:,h],ignore_index=-1)
                                  for h,g in enumerate(ax))/len(ax)
            if not torch.isfinite(L): continue
            sc.scale(L).backward(); sc.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
        pv=col(va,tta=False); au=roc_auc_score(y[va],pv) if len(set(y[va]))>1 else 0
        if au>best: best=au; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0; ep_best=ep
        else: ni+=1
        if ni>=PATIENCE: break
    net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
    p=col(te); a=roc_auc_score(y[te],p)
    runs.append((a,p))
    print("  seed "+str(sd)+"  test AUC "+format(a,".4f")+
          "   best epoch "+str(ep_best)+"   "+format(time.time()-t0,".0f")+"s")

aucs=[r[0] for r in runs]
mi=int(np.argsort(aucs)[len(aucs)//2])          # MEDIAN, not max
print("\n  seeds: "+str([round(a,4) for a in aucs]))
print("  MEDIAN "+format(aucs[mi],".4f")+"   (using this)")

OOF["pred"][te]=runs[mi][1]
d["prob_pred"]=OOF["pred"]
d.to_csv(os.path.join(D,"phase3_mass.csv"),index=False)

from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
print("\n"+"="*66); print("MASS — updated pooled result"); print("="*66)
for cond,tag in [("ref","REFERENCE masks"),("pred","PREDICTED masks")]:
    L=d.groupby("lesion_key").agg(yy=("true","max"),pp=("prob_"+cond,"mean")).reset_index()
    thr=max([(balanced_accuracy_score(L.yy,(L.pp>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    pr=(L.pp>thr).astype(int); tn,fp,fn,tp=confusion_matrix(L.yy,pr,labels=[0,1]).ravel()
    print("  "+tag.ljust(20)+"per-lesion AUC "+format(roc_auc_score(L.yy,L.pp),".4f")+
          "  acc "+format(100*accuracy_score(L.yy,pr),".1f")+"%  FP "+str(fp)+" FN "+str(fn))
Lr=d.groupby("lesion_key").agg(yy=("true","max"),r=("prob_ref","mean"),q=("prob_pred","mean")).reset_index()
print("\n  cost of predicted masks: "+format(roc_auc_score(Lr.yy,Lr.q)-roc_auc_score(Lr.yy,Lr.r),"+.4f"))
print("="*66)

fold 3  train 1195  val 162  test 339
original predicted-mask AUC: 0.7213  (492s)
reference-mask AUC:          0.8497

  seed 103  test AUC 0.8490   best epoch 11   1907s
  seed 203  test AUC 0.8510   best epoch 10   1797s
  seed 303  test AUC 0.7428   best epoch 2   938s

  seeds: [0.849, 0.851, 0.7428]
  MEDIAN 0.8490   (using this)

MASS — updated pooled result
  REFERENCE masks     per-lesion AUC 0.8441  acc 78.2%  FP 119 FN 100
  PREDICTED masks     per-lesion AUC 0.8449  acc 78.5%  FP 74 FN 142

  cost of predicted masks: +0.0008


In [10]:
# ══════════════════════════════════════════════════════════════════════
# Add the end-to-end predictions to the mass ensemble
# ══════════════════════════════════════════════════════════════════════
import os, itertools, numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
D="/root/autodl-tmp/CBIS"

SRC={"separate":("cv_mass_fixed_oof.csv",None),
     "joint":("cv_joint_oof.csv",1),
     "joint_soft":("cv_joint_soft_oof.csv",1),
     "dualpath":("cv_mass_dualpath_oof.csv",None),
     "imageonly":("cv_mass_imageonly_oof.csv",None),
     "endtoend":("phase3_mass.csv",None)}      # NEW - predicted-mask model

base=None; P={}
for n,(f,lt) in SRC.items():
    p=os.path.join(D,f)
    if not os.path.exists(p): print("missing "+f); continue
    x=pd.read_csv(p)
    if lt is not None and "ltype" in x.columns: x=x[x.ltype==lt]
    yc="true" if "true" in x.columns else "label"
    pc="prob_pred" if n=="endtoend" else "prob"
    x=x[["img",yc,pc]].rename(columns={yc:"y",pc:"pr"})
    if base is None:
        src=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv"))
        sc=[c for c in ["side","left or right breast"] if c in src.columns][0]
        lc=[c for c in ["lesion","abnormality id"] if c in src.columns][0]
        k=src[["img"]].copy(); k["lesion_key"]=src.patient_id.astype(str)+"_"+src[sc].astype(str)+"_"+src[lc].astype(str)
        base=x.merge(k,on="img",how="left")
    P[n]=x.set_index("img")["pr"]

base=base.set_index("img")
for n,s in P.items(): base[n]=s
base=base.dropna(subset=list(P.keys())).reset_index()
y=base.y.astype(int).values
print("models: "+str(list(P.keys()))+"   n="+str(len(base))+"\n")
for n in P: print("  "+n.ljust(12)+format(roc_auc_score(y,base[n]),".4f"))
print("\ncorrelation with endtoend:")
print(base[list(P.keys())].corr(method="spearman")["endtoend"].round(3).to_string())

def agg(df,col,how):
    if how=="mean": g=df.groupby("lesion_key").agg(y=("y","max"),p=(col,"mean"))
    else:
        t=df.copy(); t["w"]=np.abs(t[col]-0.5)+1e-3; t["wp"]=t[col]*t["w"]
        g=t.groupby("lesion_key").agg(y=("y","max"),wp=("wp","sum"),w=("w","sum"))
        g["p"]=g.wp/g.w; g=g[["y","p"]]
    return g.reset_index()
def score(g):
    yy=g.y.astype(int).values; pp=g.p.values
    thr=max([(balanced_accuracy_score(yy,(pp>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    pr=(pp>thr).astype(int); tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
    return roc_auc_score(yy,pp), accuracy_score(yy,pr), tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn

res=[]; names=list(P.keys())
for r in range(1,len(names)+1):
    for c in itertools.combinations(names,r):
        rk=np.mean([rankdata(base[n].values)/len(base) for n in c],axis=0)
        t=base[["lesion_key","y"]].copy(); t["e"]=rk
        for how in ["mean","confw"]:
            a,ac,se,sp,fp,fn=score(agg(t,"e",how))
            res.append((a,ac,se,sp,fp,fn,"+".join(c),how))
res.sort(reverse=True)
print("\n  combination                                  agg     AUC     acc    sens  spec  FP  FN")
for a,ac,se,sp,fp,fn,c,how in res[:10]:
    print("  "+c.ljust(44)+how.ljust(7)+format(a,".4f")+"  "+format(100*ac,".1f")+"%  "+
          format(se,".3f")+" "+format(sp,".3f")+"  "+str(fp).rjust(3)+" "+str(fn).rjust(3))
best=res[0]
print("\n  previous best (without endtoend): 0.8692")
print("  new best: "+format(best[0],".4f")+"  ("+format(best[0]-0.8692,"+.4f")+")")
print("  uses endtoend: "+("YES" if "endtoend" in best[6] else "no"))

models: ['separate', 'joint', 'joint_soft', 'dualpath', 'imageonly', 'endtoend']   n=1696

  separate    0.8308
  joint       0.7978
  joint_soft  0.8220
  dualpath    0.7882
  imageonly   0.7603
  endtoend    0.8252

correlation with endtoend:
separate      0.885
joint         0.746
joint_soft    0.773
dualpath      0.725
imageonly     0.689
endtoend      1.000

  combination                                  agg     AUC     acc    sens  spec  FP  FN
  separate+joint_soft                         confw  0.8692  79.9%  0.803 0.796  111  91
  separate+joint_soft                         mean   0.8686  79.6%  0.844 0.755  133  72
  separate+joint_soft+endtoend                mean   0.8682  80.5%  0.790 0.818   99  97
  separate+joint_soft+endtoend                confw  0.8679  80.3%  0.797 0.808  104  94
  separate+joint+joint_soft+dualpath+endtoend mean   0.8671  80.6%  0.810 0.803  107  88
  separate+joint_soft+dualpath+endtoend       mean   0.8669  80.4%  0.747 0.853   80 117
  separate+

In [11]:
# ══════════════════════════════════════════════════════════════════════
# END-TO-END VARIANTS — three more predicted-mask models, then ensemble
#   ALL use predicted masks. Nothing uses reference masks.
#   V1: soft labels (BI-RADS-conditioned)
#   V2: attention weight 1+3*mask (stronger lesion focus)
#   V3: no aux heads, different seed
#   Then ensembles them with your existing end-to-end model.
#   Requires the Phase 3 cell to have been run (uses d, CACHE, DS, Net, meta, aux, AK, y)
# ══════════════════════════════════════════════════════════════════════
import os, time, itertools, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
D="/root/autodl-tmp/CBIS"

# soft targets from BI-RADS malignancy rate (training only)
prior=d.groupby("assessment")["label"].mean()
def softt(r):
    p=prior.get(r["assessment"],0.5); a=0.40*(1-2*abs(p-0.5))
    return (1-a)*float(r["label"])+a*p
d["soft"]=d.apply(softt,axis=1).astype(np.float32)

class NetW(nn.Module):
    def __init__(s,meta,att=2.0,use_aux=True):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features; s.att=att; s.use_aux=use_aux
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        s.keys=sorted(meta.keys())
        s.aux=nn.ModuleList([nn.Sequential(nn.Linear(1024,128),nn.ReLU(),nn.Dropout(0.3),
                                           nn.Linear(128,meta[k])) for k in s.keys]) if use_aux else nn.ModuleList()
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+s.att*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        return s.head(g),[h(g) for h in s.aux]

def train_variant(tag, att, use_aux, use_soft, seed_off):
    oof=np.zeros(len(d))
    for k in range(5):
        role=d["role_f"+str(k)]
        tr=np.where(role=="train")[0]; va=np.where(role=="val")[0]; te=np.where(role=="test")[0]
        assert not (set(d.patient_id[tr])&set(d.patient_id[te])), "LEAK"
        t0=time.time(); torch.manual_seed(seed_off+k); np.random.seed(seed_off+k)
        n0=float((y[tr]==0).sum()); n1=float((y[tr]==1).sum())
        w0=n1/(n0+n1); w1=n0/(n0+n1)
        al=torch.tensor([w0,w1],device=DEV)
        soft_t=torch.tensor(d["soft"].values,dtype=torch.float32,device=DEV)
        def lossf(lo,t2,idxb):
            if use_soft:
                lp=F.log_softmax(lo.float(),1); p1=lp[:,1].exp()
                ts=soft_t[idxb]
                ce=-(ts*lp[:,1]*w1+(1-ts)*lp[:,0]*w0)
                pt=ts*p1+(1-ts)*(1-p1)
                return ((1-pt)**GAMMA*ce).mean()
            ce=F.cross_entropy(lo.float(),t2,weight=al,reduction="none"); pt=torch.exp(-ce)
            return ((1-pt)**GAMMA*ce).mean()
        net=NetW(meta,att,use_aux).to(DEV).to(memory_format=torch.channels_last)
        for p_ in net.b.parameters(): p_.requires_grad=False
        sc=torch.amp.GradScaler()
        opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)

        class DS2(DS):
            def __getitem__(s2,i):
                out=DS.__getitem__(s2,i)
                j=s2.idx[i%len(s2.idx)]
                return out+(torch.tensor(int(j)),)
        tl=DataLoader(DS2(tr,True,True),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)

        @torch.no_grad()
        def col(idx):
            net.eval(); tot=None
            for t in [0,1,2,3]:
                ld=DataLoader(DS(idx,True,False,tta=t),batch_size=20,shuffle=False,num_workers=0); ps=[]
                for x,m,_,_ in ld:
                    x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                    with torch.amp.autocast(device_type="cuda"): o,_=net(x,m)
                    ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
                ps=np.array(ps); tot=ps if tot is None else tot+ps
            return tot/4
        best=0;bs=None;ni=0
        for ep in range(1,EPOCHS+1):
            if ep==FREEZE+1:
                for p_ in net.b.parameters(): p_.requires_grad=True
                opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
            net.train()
            if ep<=FREEZE: net.b.eval()
            for x,m,t2,a,jb in tl:
                x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                t2=t2.to(DEV); a=a.to(DEV); jb=jb.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"):
                    o,ax=net(x,m); L=lossf(o,t2,jb)
                    if len(ax):
                        L=L+AUX_W*sum(F.cross_entropy(g.float(),a[:,h],ignore_index=-1)
                                      for h,g in enumerate(ax))/len(ax)
                if not torch.isfinite(L): continue
                sc.scale(L).backward(); sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
            pv=col(va); au=roc_auc_score(y[va],pv) if len(set(y[va]))>1 else 0
            if au>best: best=au; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0
            else: ni+=1
            if ni>=6: break
        net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
        oof[te]=col(te)
        print("    fold "+str(k)+"  AUC "+format(roc_auc_score(y[te],oof[te]),".4f")+
              "  ("+format(time.time()-t0,".0f")+"s)")
    print("  "+tag+" pooled AUC "+format(roc_auc_score(y,oof),".4f"))
    return oof

print("### V1 soft labels, predicted masks");      v1=train_variant("V1",2.0,True, True, 500)
print("### V2 attention 1+3m, predicted masks");   v2=train_variant("V2",3.0,True, False,600)
print("### V3 no aux heads, predicted masks");     v3=train_variant("V3",2.0,False,False,700)

E=d[["img","lesion_key","true"]].copy()
E["e0"]=d["prob_pred"].values; E["e1"]=v1; E["e2"]=v2; E["e3"]=v3
E.to_csv(os.path.join(D,"endtoend_variants_mass.csv"),index=False)
yy=E.true.astype(int).values
print("\nindividual (all predicted-mask):")
for c in ["e0","e1","e2","e3"]: print("  "+c+"  "+format(roc_auc_score(yy,E[c]),".4f"))
print("\ncorrelation:"); print(E[["e0","e1","e2","e3"]].corr(method="spearman").round(3).to_string())

def agg(t,col,how):
    if how=="mean": g=t.groupby("lesion_key").agg(y=("true","max"),p=(col,"mean"))
    else:
        q=t.copy(); q["w"]=np.abs(q[col]-0.5)+1e-3; q["wp"]=q[col]*q["w"]
        g=q.groupby("lesion_key").agg(y=("true","max"),wp=("wp","sum"),w=("w","sum"))
        g["p"]=g.wp/g.w; g=g[["y","p"]]
    return g.reset_index()

res=[]
for r in range(1,5):
    for c in itertools.combinations(["e0","e1","e2","e3"],r):
        rk=np.mean([rankdata(E[n].values)/len(E) for n in c],axis=0)
        t=E[["lesion_key","true"]].copy(); t["x"]=rk
        for how in ["mean","confw"]:
            g=agg(t,"x",how); yv=g.y.astype(int).values; pv=g.p.values
            thr=max([(balanced_accuracy_score(yv,(pv>q).astype(int)),q) for q in np.linspace(.05,.95,181)])[1]
            pr=(pv>thr).astype(int); tn,fp,fn,tp=confusion_matrix(yv,pr,labels=[0,1]).ravel()
            res.append((roc_auc_score(yv,pv),accuracy_score(yv,pr),tp/max(tp+fn,1),tn/max(tn+fp,1),fp,fn,"+".join(c),how))
res.sort(reverse=True)
print("\n  END-TO-END ENSEMBLE (predicted masks only)")
print("  combination          agg      AUC     acc    sens  spec  FP  FN")
for a,ac,se,sp,fp,fn,c,how in res[:8]:
    print("  "+c.ljust(20)+how.ljust(7)+format(a,".4f")+"  "+format(100*ac,".1f")+"%  "+
          format(se,".3f")+" "+format(sp,".3f")+"  "+str(fp).rjust(3)+" "+str(fn).rjust(3))
print("\n  single end-to-end model was: 0.8449")
print("  best end-to-end ensemble:    "+format(res[0][0],".4f")+"  ("+format(res[0][0]-0.8449,"+.4f")+")")

### V1 soft labels, predicted masks
    fold 0  AUC 0.7790  (1690s)
    fold 1  AUC 0.8727  (1368s)
    fold 2  AUC 0.8098  (2095s)
    fold 3  AUC 0.8553  (1683s)
    fold 4  AUC 0.8583  (1427s)
  V1 pooled AUC 0.8293
### V2 attention 1+3m, predicted masks
    fold 0  AUC 0.7770  (1499s)
    fold 1  AUC 0.8722  (1963s)
    fold 2  AUC 0.7946  (1422s)
    fold 3  AUC 0.8404  (2002s)
    fold 4  AUC 0.8436  (2097s)
  V2 pooled AUC 0.8255
### V3 no aux heads, predicted masks
    fold 0  AUC 0.7611  (2135s)
    fold 1  AUC 0.8393  (1284s)
    fold 2  AUC 0.7844  (1530s)
    fold 3  AUC 0.7329  (847s)
    fold 4  AUC 0.8153  (2097s)
  V3 pooled AUC 0.7699

individual (all predicted-mask):
  e0  0.8252
  e1  0.8293
  e2  0.8255
  e3  0.7699

correlation:
       e0     e1     e2     e3
e0  1.000  0.913  0.922  0.734
e1  0.913  1.000  0.897  0.761
e2  0.922  0.897  1.000  0.719
e3  0.734  0.761  0.719  1.000

  END-TO-END ENSEMBLE (predicted masks only)
  combination          agg      AUC    

In [3]:
import os, numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv"))
PM=os.path.join(D,"predmasks_mass")
d["pred"]=d["img"].apply(lambda p: os.path.join(PM,os.path.basename(p).replace("_img.png","")+"_pred.png"))

def dice_np(m,g):
    tp=(m&g).sum(); fp=(m&~g).sum(); fn=((~m)&g).sum()
    return (2*tp+1)/(2*tp+fp+fn+1)

vals=[]
for _,r in d.iterrows():
    g=cv2.imread(str(r["msk"]),cv2.IMREAD_GRAYSCALE)
    p=cv2.imread(str(r["pred"]),cv2.IMREAD_GRAYSCALE)
    if g is None or p is None: vals.append(np.nan); continue
    gg=cv2.resize(g,(256,256),interpolation=cv2.INTER_NEAREST)>127
    pp=cv2.resize(p,(256,256),interpolation=cv2.INTER_NEAREST)>127
    vals.append(dice_np(pp,gg))
d["oof_dice"]=vals
d.to_csv(os.path.join(D,"unified_folds_mass.csv"),index=False)

folds=[d[d.fold==k]["oof_dice"].mean() for k in range(5)]
print("per fold: "+str([round(f,4) for f in folds]))
print("mean "+format(np.mean(folds),".4f")+" ± "+format(np.std(folds),".4f")+"   (was 0.8933 pre-TTA)")

per fold: [np.float64(0.8952), np.float64(0.893), np.float64(0.9037), np.float64(0.9074), np.float64(0.8999)]
mean 0.8998 ± 0.0053   (was 0.8933 pre-TTA)


In [8]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1 — MASS SEGMENTATION, full range (weaker methods included)
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","comparison"); os.makedirs(FIG,exist_ok=True)
plt.rcParams.update({"font.size":10,"figure.dpi":150})

MINE=0.900; SD=0.006
# tier: -1 you | 0 verified patient/case-level | 1 unclear | 2 likely leakage
SEG=[
 ("Bi-CBMSegNet 2025",          0.9709, 2),
 ("BiLoG-Net 2026 (preprint)",  0.9420, 0),
 ("Jalalian 2025",              0.9292, 1),
 ("Connected-SegNets 2022",     0.9286, 1),
 ("Ma & Peng 2024",             0.9246, 1),
 ("HTU-Net 2024",               0.9200, 1),
 ("TrEnD 2023",                 0.9200, 1),
 ("THIS WORK",                  MINE,  -1),
 ("Connected-UNets 2021",       0.8952, 1),
 ("YOLOv5+DW-SegNet 2025",      0.8940, 0),
 ("UNet++ 2025",                0.8654, 1),
 ("PSPNet 2025",                0.8410, 1),
 ("AUNet 2020",                 0.8180, 1),
 ("TransUNet (mammo) 2025",     0.8050, 1),
 ("Standard U-Net baseline",    0.7890, 1),
 ("Tsochatzidis 2021",          0.7220, 0),
 ("SegNet baseline",            0.6910, 1),
 ("Tiryaki 2023",               0.6356, 1),
 ("FCN baseline",               0.6120, 1),
]
SEG.sort(key=lambda r:-r[1])
TIER={-1:"patient-grouped 5-fold CV",0:"patient / case-level",
      1:"ROI / image-wise / unclear",2:"LIKELY LEAKAGE"}
COL={-1:"#1f6fb4",0:"#4a9d5f",1:"#b0aea6",2:"#d4a5a5"}

W=84
print("="*W); print("MASS SEGMENTATION — full ranking"); print("="*W)
print(f"{'#':<4}{'Study':<32}{'Dice':<10}{'Split protocol'}")
print("-"*W)
for i,(n,v,t) in enumerate(SEG,1):
    print(f"{i:<4}{n:<32}{v:<10.4f}{TIER[t]}"+("   <<<" if t==-1 else ""))
rank=[i for i,(n,_,_) in enumerate(SEG,1) if n=="THIS WORK"][0]
print("-"*W)
print("  your rank: "+str(rank)+" of "+str(len(SEG))+"   |  "+
      str(len(SEG)-rank)+" methods below you")
print("="*W)

fig,ax=plt.subplots(figsize=(9.5,8))
nm=[r[0] for r in SEG][::-1]; vl=[r[1] for r in SEG][::-1]; tr=[r[2] for r in SEG][::-1]
b=ax.barh(range(len(vl)),vl,height=.7,color=[COL[t] for t in tr],
          edgecolor=["black" if t==-1 else "none" for t in tr],
          linewidth=[1.8 if t==-1 else 0 for t in tr])
ax.set_yticks(range(len(nm))); ax.set_yticklabels(nm,fontsize=9)
ax.set_xlim(0.55,1.02); ax.set_xlabel("Dice coefficient")
ax.set_title("Mass segmentation on CBIS-DDSM\ncolour = evaluation rigour",fontsize=12)
ax.grid(axis="x",alpha=.25); ax.set_axisbelow(True)
for bb,v,t in zip(b,vl,tr):
    ax.text(v+.005,bb.get_y()+bb.get_height()/2,format(v,".4f"),va="center",
            fontsize=8.5,fontweight="bold" if t==-1 else "normal")
ax.errorbar(MINE,nm.index("THIS WORK"),xerr=SD,color="black",capsize=4,lw=1.6,zorder=5)
ax.legend(handles=[Patch(facecolor=COL[-1],edgecolor="black",label="This work"),
                   Patch(facecolor=COL[0],label="Verified patient/case-level"),
                   Patch(facecolor=COL[1],label="ROI / image-wise / unclear"),
                   Patch(facecolor=COL[2],label="Likely data leakage")],
          loc="lower right",fontsize=9,framealpha=.95)
plt.tight_layout(); plt.savefig(os.path.join(FIG,"mass_seg_full.png"),bbox_inches="tight"); plt.close()
print("saved mass_seg_full.png")

MASS SEGMENTATION — full ranking
#   Study                           Dice      Split protocol
------------------------------------------------------------------------------------
1   Bi-CBMSegNet 2025               0.9709    LIKELY LEAKAGE
2   BiLoG-Net 2026 (preprint)       0.9420    patient / case-level
3   Jalalian 2025                   0.9292    ROI / image-wise / unclear
4   Connected-SegNets 2022          0.9286    ROI / image-wise / unclear
5   Ma & Peng 2024                  0.9246    ROI / image-wise / unclear
6   HTU-Net 2024                    0.9200    ROI / image-wise / unclear
7   TrEnD 2023                      0.9200    ROI / image-wise / unclear
8   THIS WORK                       0.9000    patient-grouped 5-fold CV   <<<
9   Connected-UNets 2021            0.8952    ROI / image-wise / unclear
10  YOLOv5+DW-SegNet 2025           0.8940    patient / case-level
11  UNet++ 2025                     0.8654    ROI / image-wise / unclear
12  PSPNet 2025                     0

In [6]:
# ══════════════════════════════════════════════════════════════════════
# MASS CLASSIFICATION — comparison with literature
#   TABLE 1: classification only (AUC / accuracy)
#   TABLE 2: two-stage pipelines (segmentation Dice + classification AUC)
#   reads your own numbers from disk where possible
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","comparison"); os.makedirs(FIG,exist_ok=True)
plt.rcParams.update({"font.size":10,"figure.dpi":150})

# ---------- your numbers ----------
def lesion_metrics(path,col,ycol=None):
    x=pd.read_csv(os.path.join(D,path))
    yc=ycol or ("true" if "true" in x.columns else "label")
    g=x.groupby("lesion_key").agg(y=(yc,"max"),p=(col,"mean")).reset_index()
    thr=max([(balanced_accuracy_score(g.y,(g.p>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    return roc_auc_score(g.y,g.p), 100*accuracy_score(g.y,(g.p>thr).astype(int))

E2E_AUC,E2E_ACC = None,None
try:
    e=pd.read_csv(os.path.join(D,"endtoend_variants_mass.csv"))
    from scipy.stats import rankdata
    rk=np.mean([rankdata(e[c].values)/len(e) for c in ["e0","e1","e2"]],axis=0)
    t=e[["lesion_key","true"]].copy(); t["x"]=rk
    t["w"]=np.abs(t.x-0.5)+1e-3; t["wp"]=t.x*t.w
    g=t.groupby("lesion_key").agg(y=("true","max"),wp=("wp","sum"),w=("w","sum")).reset_index()
    g["p"]=g.wp/g.w
    thr=max([(balanced_accuracy_score(g.y,(g.p>q).astype(int)),q) for q in np.linspace(.05,.95,181)])[1]
    E2E_AUC=roc_auc_score(g.y,g.p); E2E_ACC=100*accuracy_score(g.y,(g.p>thr).astype(int))
except Exception as ex:
    E2E_AUC,E2E_ACC = 0.8558, 79.5
    print("(using stored end-to-end values)")

SEG_DICE=None
try:
    s=pd.read_csv(os.path.join(D,"unified_folds_mass.csv"))
    SEG_DICE=float(np.mean([s[s.fold==k]["oof_dice"].mean() for k in range(5)]))
except Exception: SEG_DICE=0.900

print("THIS WORK — mass")
print("  segmentation Dice (5-fold patient CV): "+format(SEG_DICE,".4f"))
print("  classification AUC (end-to-end)      : "+format(E2E_AUC,".4f")+"   acc "+format(E2E_ACC,".1f")+"%\n")

# tier: -1 you | 0 verified patient/case-level | 1 unclear | 2 likely leakage
CLS=[
 ("Saha 2024 (SAM+PTr, INbreast)", 0.9998, 99.9, 2),
 ("BiLoG-Net 2026 (preprint)",     0.9710, 95.2, 0),
 ("MSDLM 2025",                    0.9575, 97.6, 2),
 ("Baccouche 2022",                0.9500, 95.1, 1),
 ("Ma & Peng 2024",                0.9320, None, 1),
 ("Ragab 2019",                    0.9400, 87.2, 0),
 ("Tsochatzidis 2021",             0.8620, 74.9, 0),
 ("THIS WORK (end-to-end)",        E2E_AUC, E2E_ACC, -1),
 ("Cantone 2023 (OMI-DB)",         None,   85.2, 0),
 ("Tiryaki 2023",                  0.8188, 76.2, 1),
 ("Salama 2021 (CBIS)",            None,   82.5, 1),
 ("Ansar 2020",                    None,   74.5, 1),
]
TIER={-1:"patient-grouped 5-fold CV",0:"patient / case-level",
      1:"ROI / image-wise / unclear",2:"LIKELY LEAKAGE"}

W=94
print("="*W); print("TABLE 1 — MASS CLASSIFICATION (benign vs malignant)"); print("="*W)
print(f"{'#':<4}{'Study':<32}{'AUC':<10}{'Acc':<9}{'Split protocol'}")
print("-"*W)
srt=sorted(CLS,key=lambda r:-(r[1] if r[1] is not None else r[2]/100))
for i,(n,a,ac,t) in enumerate(srt,1):
    print(f"{i:<4}{n:<32}{(format(a,'.4f') if a else '—'):<10}"
          f"{(format(ac,'.1f')+'%' if ac else '—'):<9}{TIER[t]}"+("   <<" if t==-1 else ""))
print("="*W)
fair=[r for r in srt if r[3] in (-1,0)]
print("\nVerified patient/case-level splits only:")
for i,(n,a,ac,t) in enumerate(fair,1):
    print("  "+str(i)+". "+n.ljust(32)+(format(a,".4f") if a else "—")+("   <<" if t==-1 else ""))

# ---------- TABLE 2: two-stage pipelines ----------
PIPE=[
 ("Bi-CBMSegNet 2025",      0.9709, None,   2, "segmentation only"),
 ("BiLoG-Net 2026",         0.9420, 0.9710, 0, "joint seg + classification"),
 ("Ma & Peng 2024",         0.9246, 0.9320, 1, "cross-view VAE, split unstated"),
 ("THIS WORK",              SEG_DICE, E2E_AUC, -1, "seg-guided, predicted masks"),
 ("Connected-UNets 2021",   0.8952, None,   1, "segmentation only"),
 ("YOLOv5+DW-SegNet 2025",  0.8940, None,   0, "detection + segmentation"),
 ("Tsochatzidis 2021",      0.7220, 0.8620, 0, "seg map injected into CNN"),
 ("Tiryaki 2023",           0.6356, 0.8188, 1, "cascaded U-Net++/Xception"),
]
print("\n"+"="*W); print("TABLE 2 — TWO-STAGE PIPELINES (segmentation + classification)"); print("="*W)
print(f"{'Study':<28}{'Dice':<10}{'AUC':<10}{'Split':<26}{'Note'}")
print("-"*W)
for n,dc,a,t,note in sorted(PIPE,key=lambda r:-(r[1] or 0)):
    print(f"{n:<28}{(format(dc,'.4f') if dc else '—'):<10}"
          f"{(format(a,'.4f') if a else '—'):<10}{TIER[t]:<26}{note}"+("   <<" if t==-1 else ""))
print("="*W)
both=[r for r in PIPE if r[1] and r[2]]
print("\nStudies reporting BOTH stages:")
for n,dc,a,t,note in sorted(both,key=lambda r:-r[1]):
    print("  "+n.ljust(28)+"Dice "+format(dc,".4f")+"   AUC "+format(a,".4f")+
          "   ["+TIER[t]+"]"+("   <<" if t==-1 else ""))

# ══════════════ FIGURES ══════════════
COL={-1:"#1f6fb4",0:"#4a9d5f",1:"#b0aea6",2:"#d4a5a5"}
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(15,6.2),gridspec_kw={"width_ratios":[1.5,1]})

auc_rows=[(n,a,t) for n,a,ac,t in srt if a is not None]
nm=[r[0] for r in auc_rows][::-1]; vl=[r[1] for r in auc_rows][::-1]; tr=[r[2] for r in auc_rows][::-1]
b=ax1.barh(range(len(vl)),vl,height=.66,color=[COL[t] for t in tr],
           edgecolor=["black" if t==-1 else "none" for t in tr],
           linewidth=[1.5 if t==-1 else 0 for t in tr])
ax1.set_yticks(range(len(nm))); ax1.set_yticklabels(nm,fontsize=9)
ax1.set_xlim(0.75,1.03); ax1.set_xlabel("AUC")
ax1.set_title("Mass classification — all reported AUCs\ncolour = evaluation rigour",fontsize=11)
ax1.grid(axis="x",alpha=.25); ax1.set_axisbelow(True)
for bb,v,t in zip(b,vl,tr):
    ax1.text(v+.004,bb.get_y()+bb.get_height()/2,format(v,".4f"),va="center",
             fontsize=8.5,fontweight="bold" if t==-1 else "normal")
ax1.legend(handles=[Patch(facecolor=COL[-1],edgecolor="black",label="This work"),
                    Patch(facecolor=COL[0],label="Verified patient/case-level"),
                    Patch(facecolor=COL[1],label="ROI / image-wise / unclear"),
                    Patch(facecolor=COL[2],label="Likely data leakage")],
           loc="lower right",fontsize=8.5,framealpha=.95)

fa=[(n,a,t) for n,a,ac,t in fair if a is not None]
fn=[r[0] for r in fa][::-1]; fv=[r[1] for r in fa][::-1]; ft=[r[2] for r in fa][::-1]
b2=ax2.barh(range(len(fv)),fv,height=.5,color=[COL[t] for t in ft],
            edgecolor=["black" if t==-1 else "none" for t in ft],
            linewidth=[1.5 if t==-1 else 0 for t in ft])
ax2.set_yticks(range(len(fn))); ax2.set_yticklabels(fn,fontsize=9)
ax2.set_xlim(0.78,1.0); ax2.set_xlabel("AUC")
ax2.set_title("Like-for-like\nverified patient/case-level splits",fontsize=11)
ax2.grid(axis="x",alpha=.25); ax2.set_axisbelow(True)
for bb,v in zip(b2,fv): ax2.text(v+.004,bb.get_y()+bb.get_height()/2,format(v,".4f"),va="center",fontsize=8.5)
plt.tight_layout(); plt.savefig(os.path.join(FIG,"mass_classification_comparison.png"),bbox_inches="tight"); plt.close()

# scatter: Dice vs AUC for two-stage pipelines
fig,ax=plt.subplots(figsize=(8,6.4))
for n,dc,a,t,note in both:
    ax.scatter(dc,a,s=210 if t==-1 else 120,color=COL[t],
               edgecolor="black",linewidth=1.5 if t==-1 else .6,zorder=3)
    ax.annotate(n,(dc,a),xytext=(7,7),textcoords="offset points",
                fontsize=9,fontweight="bold" if t==-1 else "normal")
ax.set_xlabel("Segmentation Dice"); ax.set_ylabel("Classification AUC")
ax.set_title("Two-stage pipelines: segmentation quality vs classification performance",fontsize=11)
ax.grid(alpha=.25); ax.set_axisbelow(True)
ax.legend(handles=[Patch(facecolor=COL[-1],edgecolor="black",label="This work"),
                   Patch(facecolor=COL[0],label="Verified split"),
                   Patch(facecolor=COL[1],label="Unclear split")],
          loc="lower right",fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(FIG,"pipeline_dice_vs_auc.png"),bbox_inches="tight"); plt.close()
print("\nfigures saved to "+FIG)

THIS WORK — mass
  segmentation Dice (5-fold patient CV): 0.8998
  classification AUC (end-to-end)      : 0.8558   acc 79.5%

TABLE 1 — MASS CLASSIFICATION (benign vs malignant)
#   Study                           AUC       Acc      Split protocol
----------------------------------------------------------------------------------------------
1   Saha 2024 (SAM+PTr, INbreast)   0.9998    99.9%    LIKELY LEAKAGE
2   BiLoG-Net 2026 (preprint)       0.9710    95.2%    patient / case-level
3   MSDLM 2025                      0.9575    97.6%    LIKELY LEAKAGE
4   Baccouche 2022                  0.9500    95.1%    ROI / image-wise / unclear
5   Ragab 2019                      0.9400    87.2%    patient / case-level
6   Ma & Peng 2024                  0.9320    —        ROI / image-wise / unclear
7   Tsochatzidis 2021               0.8620    74.9%    patient / case-level
8   THIS WORK (end-to-end)          0.8558    79.5%    patient-grouped 5-fold CV   <<
9   Cantone 2023 (OMI-DB)           —  

In [7]:
# ══════════════════════════════════════════════════════════════════════
# TWO-STAGE PIPELINE COMPARISON — grouped bars, easy to read
#   only studies that report BOTH segmentation and classification
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","comparison"); os.makedirs(FIG,exist_ok=True)
plt.rcParams.update({"font.size":11,"figure.dpi":150})

SEG_DICE=0.900        # your 5-fold patient-grouped CV
CLS_AUC =0.8558       # your end-to-end ensemble, predicted masks

# (study, seg Dice, cls AUC, split verified?, note)
P=[
 ("BiLoG-Net\n2026",        0.9420, 0.9710, True,  "preprint, unrefereed"),
 ("THIS WORK",              SEG_DICE, CLS_AUC, True, "predicted masks, end-to-end"),
 ("Ma & Peng\n2024",        0.9246, 0.9320, False, "split not stated"),
 ("Tsochatzidis\n2021",     0.7220, 0.8620, True,  "official CBIS split"),
 ("Tiryaki\n2023",          0.6356, 0.8188, False, "split not stated"),
]

x=np.arange(len(P)); w=0.36
seg=[p[1] for p in P]; cls=[p[2] for p in P]
mine=[p[0]=="THIS WORK" for p in P]

fig,ax=plt.subplots(figsize=(11,6.4))
b1=ax.bar(x-w/2,seg,w,label="Segmentation (Dice)",color="#4a9d5f",
          edgecolor=["black" if m else "none" for m in mine],
          linewidth=[2 if m else 0 for m in mine])
b2=ax.bar(x+w/2,cls,w,label="Classification (AUC)",color="#1f6fb4",
          edgecolor=["black" if m else "none" for m in mine],
          linewidth=[2 if m else 0 for m in mine])

for bars,vals in [(b1,seg),(b2,cls)]:
    for bb,v,m in zip(bars,vals,mine):
        ax.text(bb.get_x()+bb.get_width()/2, v+0.012, format(v,".3f"),
                ha="center", fontsize=10, fontweight="bold" if m else "normal")

# hatch the studies whose split isn't verified
for i,p in enumerate(P):
    if not p[3]:
        b1[i].set_hatch("///"); b2[i].set_hatch("///")

ax.set_xticks(x)
ax.set_xticklabels([p[0] for p in P],fontsize=10)
ax.set_ylim(0.55,1.06)
ax.set_ylabel("Score")
ax.set_title("Two-stage pipelines on CBIS-DDSM: segmentation and classification\n"
             "(hatched = split protocol not verified; black outline = this work)",fontsize=12)
ax.grid(axis="y",alpha=.25); ax.set_axisbelow(True)
ax.legend(loc="lower left",fontsize=10,framealpha=.95)

# annotate the key contrast
ax.annotate("", xy=(1-w/2, SEG_DICE+0.03), xytext=(3-w/2, 0.7220+0.03),
            arrowprops=dict(arrowstyle="<->",color="#4a9d5f",lw=1.6,alpha=.7))
ax.text(2-w/2, 0.845, "+0.178 Dice", ha="center", fontsize=9.5,
        color="#2d6b3d", fontweight="bold")

plt.tight_layout()
p=os.path.join(FIG,"pipeline_two_stage_bars.png")
plt.savefig(p,bbox_inches="tight"); plt.close()

print("="*72)
print("TWO-STAGE PIPELINES — both stages reported")
print("="*72)
print(f"{'Study':<22}{'Dice':<10}{'AUC':<10}{'Split verified':<16}{'Note'}")
print("-"*72)
for n,dc,a,ok,note in P:
    print(f"{n.replace(chr(10),' '):<22}{dc:<10.4f}{a:<10.4f}{('yes' if ok else 'NO'):<16}{note}")
print("="*72)
print("\nvs Tsochatzidis (the strongest verified-split comparator):")
print("  segmentation  0.900 vs 0.722   -> +0.178")
print("  classification 0.856 vs 0.862  -> -0.006")
print("\nsaved "+p)

TWO-STAGE PIPELINES — both stages reported
Study                 Dice      AUC       Split verified  Note
------------------------------------------------------------------------
BiLoG-Net 2026        0.9420    0.9710    yes             preprint, unrefereed
THIS WORK             0.9000    0.8558    yes             predicted masks, end-to-end
Ma & Peng 2024        0.9246    0.9320    NO              split not stated
Tsochatzidis 2021     0.7220    0.8620    yes             official CBIS split
Tiryaki 2023          0.6356    0.8188    NO              split not stated

vs Tsochatzidis (the strongest verified-split comparator):
  segmentation  0.900 vs 0.722   -> +0.178
  classification 0.856 vs 0.862  -> -0.006

saved /root/autodl-tmp/CBIS/figures/comparison/pipeline_two_stage_bars.png


In [9]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2 — CALCIFICATION CLASSIFICATION (no segmentation)
#   your numbers read from disk
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","comparison")
plt.rcParams.update({"font.size":10,"figure.dpi":150})

x=pd.read_csv(os.path.join(D,"phase3_calc.csv"))
yc="true" if "true" in x.columns else "label"
x["assessment"]=pd.to_numeric(x["assessment"],errors="coerce")
def met(col,sub=None):
    q=x if sub is None else x[sub]
    g=q.groupby("lesion_key").agg(y=(yc,"max"),p=(col,"mean")).reset_index()
    thr=max([(balanced_accuracy_score(g.y,(g.p>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    return roc_auc_score(g.y,g.p), 100*accuracy_score(g.y,(g.p>thr).astype(int))
A_ALL,ACC_ALL = met("prob_pred")
A_NO4,ACC_NO4 = met("prob_pred", x["assessment"]!=4)
print("THIS WORK — calcification classification (end-to-end, predicted masks)")
print("  all BI-RADS      AUC "+format(A_ALL,".4f")+"   acc "+format(ACC_ALL,".1f")+"%")
print("  BI-RADS 4 excl.  AUC "+format(A_NO4,".4f")+"   acc "+format(ACC_NO4,".1f")+"%\n")

# tier: -1 you | 0 verified | 1 unclear/image-wise | 2 flagged
CAL=[
 ("Elumalai 2026",              0.9820, 96.8, 2, "tiny test set; template-error labels"),
 ("Singh 2022",                 0.9600, 94.0, 1, "image-wise ROI split"),
 ("THIS WORK (B4 excluded)",    A_NO4,  ACC_NO4, -1, "Shia protocol, film"),
 ("Shia 2025",                  0.9300, 86.9, 0, "B4 excluded, FFDM"),
 ("Liu 2021 (image+clinical)",  0.9100, None, 0, "B4 only, FFDM + clinical"),
 ("Lin 2025",                   0.8880, 84.6, 0, "case-level, spot magnification"),
 ("DBT ensemble CNN 2021",      0.8837, 82.0, 1, "tomosynthesis"),
 ("Liu 2021 (image only)",      0.8410, None, 0, "B4 only, FFDM"),
 ("Cantone 2023",               None,   85.2, 0, "OMI-DB, 33 models"),
 ("Stelzer (radiomics)",        0.8250, None, 0, "B4 only, FFDM"),
 ("Multi-scale patch 2023",     0.8090, None, 0, "official CBIS split"),
 ("THIS WORK (all BI-RADS)",    A_ALL,  ACC_ALL, -1, "full difficulty retained, film"),
 ("Radiomics LDA 2025",         0.6828, None, 0, "external validation"),
]
TIER={-1:"patient-grouped 5-fold CV",0:"patient / case-level",
      1:"image-wise / unclear",2:"FLAGGED"}
COL={-1:"#1f6fb4",0:"#4a9d5f",1:"#b0aea6",2:"#d4a5a5"}
CAL.sort(key=lambda r:-(r[1] if r[1] is not None else 0))

W=100
print("="*W); print("CALCIFICATION CLASSIFICATION — ranked by AUC"); print("="*W)
print(f"{'#':<4}{'Study':<30}{'AUC':<10}{'Acc':<9}{'Split':<28}{'Note'}")
print("-"*W)
for i,(n,a,ac,t,note) in enumerate(CAL,1):
    print(f"{i:<4}{n:<30}{(format(a,'.4f') if a else '—'):<10}"
          f"{(format(ac,'.1f')+'%' if ac else '—'):<9}{TIER[t]:<28}{note[:22]}"+
          ("   <<<" if t==-1 else ""))
print("="*W)
r1=[i for i,(n,*_) in enumerate(CAL,1) if n=="THIS WORK (B4 excluded)"][0]
r2=[i for i,(n,*_) in enumerate(CAL,1) if n=="THIS WORK (all BI-RADS)"][0]
print("  B4-excluded result: rank "+str(r1)+" of "+str(len(CAL)))
print("  full-data result:   rank "+str(r2)+" of "+str(len(CAL)))
print("\nDirect protocol match — Shia et al. 2025 also excludes BI-RADS 4:")
print("  Shia (FFDM):        AUC ~0.930   acc 86.9%")
print("  YOU  (scanned film): AUC "+format(A_NO4,".4f")+"   acc "+format(ACC_NO4,".1f")+"%")

fig,ax=plt.subplots(figsize=(10,7.5))
rows=[r for r in CAL if r[1] is not None]
nm=[r[0] for r in rows][::-1]; vl=[r[1] for r in rows][::-1]; tr=[r[3] for r in rows][::-1]
b=ax.barh(range(len(vl)),vl,height=.7,color=[COL[t] for t in tr],
          edgecolor=["black" if t==-1 else "none" for t in tr],
          linewidth=[1.8 if t==-1 else 0 for t in tr])
ax.set_yticks(range(len(nm))); ax.set_yticklabels(nm,fontsize=9)
ax.set_xlim(0.62,1.02); ax.set_xlabel("AUC")
ax.set_title("Calcification classification (benign vs malignant)\ncolour = evaluation rigour",fontsize=12)
ax.grid(axis="x",alpha=.25); ax.set_axisbelow(True)
for bb,v,t in zip(b,vl,tr):
    ax.text(v+.005,bb.get_y()+bb.get_height()/2,format(v,".4f"),va="center",
            fontsize=8.5,fontweight="bold" if t==-1 else "normal")
ax.legend(handles=[Patch(facecolor=COL[-1],edgecolor="black",label="This work"),
                   Patch(facecolor=COL[0],label="Verified patient/case-level"),
                   Patch(facecolor=COL[1],label="Image-wise / unclear"),
                   Patch(facecolor=COL[2],label="Flagged (unreliable)")],
          loc="lower right",fontsize=9,framealpha=.95)
plt.tight_layout(); plt.savefig(os.path.join(FIG,"calc_classification.png"),bbox_inches="tight"); plt.close()
print("\nsaved calc_classification.png")

THIS WORK — calcification classification (end-to-end, predicted masks)
  all BI-RADS      AUC 0.7733   acc 70.4%
  BI-RADS 4 excl.  AUC 0.8880   acc 82.9%

CALCIFICATION CLASSIFICATION — ranked by AUC
#   Study                         AUC       Acc      Split                       Note
----------------------------------------------------------------------------------------------------
1   Elumalai 2026                 0.9820    96.8%    FLAGGED                     tiny test set; templat
2   Singh 2022                    0.9600    94.0%    image-wise / unclear        image-wise ROI split
3   Shia 2025                     0.9300    86.9%    patient / case-level        B4 excluded, FFDM
4   Liu 2021 (image+clinical)     0.9100    —        patient / case-level        B4 only, FFDM + clinic
5   THIS WORK (B4 excluded)       0.8880    82.9%    patient-grouped 5-fold CV   Shia protocol, film   <<<
6   Lin 2025                      0.8880    84.6%    patient / case-level        case-level, spo